<p class="lead">Este notebook amplía la séptima unidad del curso, en el lenguaje R que usan los laboratorios. Las diapositivas de clase son un mapa; este texto es el territorio. Las Unidades 2–6 trataron sobre el <em>aprendizaje</em>: dados unos datos, ajustar un modelo que prediga. Esta unidad trata sobre la otra mitad de todo sistema de aprendizaje, la que ha estado escondida dentro del paso <code>fit</code> desde la Unidad 2: la <em>optimización</em>, la búsqueda de los parámetros, el subconjunto de características, la arquitectura o la molécula que hace que algún objetivo sea lo más grande posible. El descenso de gradiente, el optimizador de las Unidades 2 y 6, necesita un objetivo suave y una derivada. Muchísimos problemas de la bioingeniería no tienen ni lo uno ni lo otro: qué características conservar, en qué orden visitar un conjunto de puntos, qué combinación de ángulos de haz entrega una dosis de radiación, qué grafo molecular se une a un receptor. Los <strong>algoritmos genéticos</strong> (GA, por sus siglas en inglés) abordan estos problemas imitando la selección natural: se evalúa una población de soluciones candidatas, se seleccionan las mejores, se recombinan y se mutan, y el ciclo se repite hasta que emergen buenas soluciones. La unidad define el método, disecciona cada uno de sus componentes con código, sigue a mano el ejemplo clásico de Goldberg, construye una implementación reutilizable y la prueba en tres problemas de referencia, enuncia la teoría que supuestamente explica por qué funciona (y las críticas a esa teoría), ubica al GA entre sus parientes —estrategias evolutivas, programación genética, evolución diferencial, enjambres de partículas, recocido simulado— y cierra con las dos aplicaciones que encontrarás con más frecuencia en aprendizaje automático, la selección de características y el ajuste de hiperparámetros, junto con las aplicaciones biomédicas documentadas en la literatura.</p>

<div class="box key"><h4>Objetivos de aprendizaje</h4>
<ul class="objectives">
<li>Explicar por qué la optimización es un pilar de la IA distinto del aprendizaje, y nombrar cuatro propiedades de una función objetivo (discreta, no diferenciable, accidentada, de caja negra) que hacen fracasar a los métodos de gradiente.</li>
<li>Definir un algoritmo genético como una búsqueda estocástica basada en poblaciones y guiada por una función de aptitud, y asignar al vocabulario biológico (gen, cromosoma, genotipo, fenotipo, población, generación, aptitud, selección, cruce, mutación) su significado algorítmico.</li>
<li>Identificar, en cualquier GA publicado, sus ocho componentes —codificación, función de aptitud, selección, cruce, mutación, reemplazo, terminación y parámetros— y describir al menos dos alternativas para cada uno.</li>
<li>Seguir a mano una generación completa del ejemplo \(f(x) = x^2\) de Goldberg: tabla de aptitudes, conteos esperados, selección por ruleta, cruce de un punto y mutación.</li>
<li>Implementar un GA completo en R base para codificaciones binarias, de valores reales y de permutaciones, y aplicarlo a una función continua multimodal, a un problema de la mochila y a una instancia del problema del viajante; compararlo con la búsqueda aleatoria, la evolución diferencial y el descenso de gradiente.</li>
<li>Enunciar el teorema de los esquemas y la hipótesis de los bloques de construcción, explicar su intuición, resumir las principales objeciones y medir la diversidad de la población en código.</li>
<li>Describir las estrategias evolutivas y CMA-ES, la programación genética, la evolución diferencial, la optimización por enjambre de partículas y el recocido simulado, y explicar lo que el teorema de no hay almuerzo gratis (no-free-lunch) dice y lo que no dice.</li>
<li>Usar un GA como envoltorio (wrapper) para la selección de características y para el ajuste de hiperparámetros, y explicar por qué la aptitud de la solución seleccionada es una estimación con sesgo optimista que debe comprobarse en un conjunto de prueba reservado (Suplemento S1).</li>
</ul></div>

<div class="box bridge"><h4>Cómo leer este notebook</h4>
<p>Cada bloque de código (chunk) es breve y está comentado. Ejecútalo, lee su salida y luego lee el párrafo que interpreta esa salida. Todo se ejecuta en un entorno de ejecución de R de Google Colab; en un entorno nuevo, instala los paquetes una sola vez con <code>install.packages(c("e1071","randomForest","rpart.plot","neuralnet","cluster","mlbench","pROC"))</code>. Esta unidad necesita solo R base y los paquetes <code>mlbench</code>, <code>class</code>, <code>e1071</code> y <code>MASS</code>, por lo que no se descarga ningún dato. <strong>No se usa ningún paquete de algoritmos genéticos</strong>: el GA se escribe desde cero en R base, exactamente como la edición en Python lo escribió desde cero; ese es el objetivo de la unidad. Todo sorteo aleatorio tiene una semilla fija, de modo que los números citados son los que verás; pero un GA es un algoritmo estocástico, y lo primero que debes hacer después de leer un resultado es cambiar la semilla y ver cuánto se mueve. Un algoritmo se sigue a mano (Sección 4) antes de que el código lo confirme.</p></div>



In [ ]:
# Bibliotecas usadas en toda la unidad. Las semillas aleatorias están fijas para que cada número citado sea reproducible.
suppressPackageStartupMessages({
  library(mlbench); library(class); library(e1071); library(MASS)
})
set.seed(0)
BLUE <- "#2b5d9b"; ORANGE <- "#d2691e"; GREEN <- "#3a7d44"; GREY <- "#8a8a8a"
# Dos funciones auxiliares usadas en todas partes: un vector de bits a cadena, y binario (bit más significativo primero) a entero.
bits2str <- function(v) paste(v, collapse = "")
decode   <- function(P) { P <- rbind(P); L <- ncol(P); as.vector(P %*% 2^((L - 1):0)) }
cat(R.version.string, "| mlbench", as.character(packageVersion("mlbench")),
    "| e1071", as.character(packageVersion("e1071")), "| class", as.character(packageVersion("class")), "\n")



La salida registra las versiones de R y de los paquetes con que se construyeron estos apuntes. Toda la unidad se apoya en R base y en los cuatro paquetes anteriores;<sup class="c"><a href="#r36">[36]</a></sup><sup class="c"><a href="#r37">[37]</a></sup> las funciones de modelado (<code>knn</code>, <code>svm</code>) provienen del conjunto estándar de herramientas de aprendizaje automático de R.<sup class="c"><a href="#r38">[38]</a></sup>

## 1. Introducción: la optimización y la búsqueda como tercer pilar

### 1.1 Aprendizaje, inferencia y optimización

Russell y Norvig abren su libro de texto con la <em>resolución de problemas mediante búsqueda</em> antes de cualquier capítulo sobre aprendizaje, y por una razón: casi todos los métodos de este curso son, en el fondo, una búsqueda. La regresión logística (Unidad 2) busca el vector de pesos que minimiza la pérdida de entropía cruzada; un árbol de decisión (Unidad 4) busca de forma voraz la división que más reduce la impureza; k-means (Unidad 5) busca los centroides que minimizan la varianza intra-grupo; una red neuronal (Unidad 6) busca sus pesos mediante descenso de gradiente. Los tres pilares de un sistema inteligente son, entonces, la <strong>representación</strong> (cómo se codifican el conocimiento y los datos — Unidad 1), la <strong>inferencia y el aprendizaje</strong> (cómo se extraen conclusiones y modelos de los datos — Unidades 2–6) y la <strong>optimización y la búsqueda</strong> (cómo se encuentra el mejor candidato dentro de un espacio de alternativas). La Unidad 1 enumeró cuatro enfoques de la IA —simbólico, conexionista, estadístico y evolutivo— y esta unidad es la evolutiva: aquí el objeto de estudio es la búsqueda misma, más que el modelo.

### 1.2 Cuándo fallan los métodos de gradiente

Los optimizadores de las Unidades 2 y 6 siguen la misma receta: calcular el gradiente del objetivo con respecto a los parámetros, dar un pequeño paso cuesta abajo, repetir. La receta es extraordinariamente eficaz cuando se cumplen sus supuestos e inútil cuando no se cumplen. Cuatro propiedades de una función objetivo la quiebran:

- **Espacios de búsqueda discretos.** Un subconjunto de características es un vector binario; una ruta por un conjunto de puntos es una permutación; un plan de tratamiento selecciona un conjunto de ángulos de haz. No existe gradiente con respecto a un bit, y no hay "pasos pequeños" entre permutaciones.
- **Objetivos no diferenciables o constantes por tramos.** La exactitud de un clasificador en función de sus hiperparámetros es una función escalonada: cambiar el \(C\) de una SVM en una cantidad diminuta normalmente no cambia ninguna predicción, así que la derivada es cero casi en todas partes.
- **Paisajes accidentados (multimodales).** Incluso cuando el objetivo es suave, el descenso de gradiente se detiene en el primer mínimo local que encuentra. En un paisaje con cientos de óptimos locales, el resultado depende por completo del punto de partida.
- **Objetivos de caja negra.** Cuando el objetivo es la salida de una simulación, de un ensayo de laboratorio o de una puntuación clínica, no hay fórmula que derivar; la única operación disponible es "evaluar este candidato".

El bloque de código siguiente muestra tres funciones objetivo. La primera es el cuenco convexo para el que está diseñado el descenso de gradiente. La segunda es una función unidimensional accidentada (un corte de la función de Rastrigin de la Sección 5); sobre ella se ejecuta el descenso de gradiente desde doce puntos de partida aleatorios y se marcan los doce puntos finales. La tercera es un objetivo discreto —la exactitud en validación cruzada de un clasificador en función de cuántas de las características mejor clasificadas usa— al que la noción de gradiente ni siquiera se aplica.



In [ ]:
rastrigin_1d <- function(x) 10 + x^2 - 10 * cos(2 * pi * x)
xs <- seq(-5.12, 5.12, length.out = 600)
set.seed(0); starts <- runif(12, -5.12, 5.12)
ends <- sapply(starts, function(s)
  optim(s, function(v) rastrigin_1d(v), method = "L-BFGS-B", lower = -5.12, upper = 5.12)$par)

# Datos de cáncer de mama (mlbench): el sustituto en R de load_breast_cancer de sklearn. 9 características nucleares.
data(BreastCancer, package = "mlbench")
bc <- BreastCancer[, -1]; bc <- bc[complete.cases(bc), ]
bc[, 1:9] <- lapply(bc[, 1:9], function(z) as.numeric(as.character(z)))
X_all <- as.matrix(bc[, 1:9]); y_all <- bc$Class
set.seed(0); n <- nrow(X_all); tr_idx <- sample(n, round(0.7 * n))
Xc_tr <- X_all[tr_idx, ]; yc_tr <- y_all[tr_idx]; Xc_te <- X_all[-tr_idx, ]; yc_te <- y_all[-tr_idx]

cv_knn_acc <- function(X, y, folds = 5, kk = 5) {       # exactitud en CV de 5 particiones de un k-NN, estandarizado dentro de cada partición
  X <- as.matrix(X); set.seed(0); fid <- sample(rep(1:folds, length.out = nrow(X)))
  mean(sapply(1:folds, function(fo) {
    tr <- fid != fo; sc <- scale(X[tr, , drop = FALSE])
    ctr <- attr(sc, "scaled:center"); scl <- attr(sc, "scaled:scale"); scl[scl == 0] <- 1
    Xte <- scale(X[!tr, , drop = FALSE], center = ctr, scale = scl)
    mean(knn(sc, Xte, y[tr], k = kk) == y[!tr])
  }))
}
Fstat <- sapply(1:9, function(j) summary(aov(Xc_tr[, j] ~ yc_tr))[[1]][1, "F value"])
ord_feat <- order(Fstat, decreasing = TRUE)
acc_k <- sapply(1:9, function(k) cv_knn_acc(Xc_tr[, ord_feat[1:k], drop = FALSE], yc_tr))

par(mfrow = c(1, 3), mar = c(4, 4, 2.4, 1))
plot(xs, xs^2, type = "l", col = BLUE, lwd = 2, xlab = "x", ylab = "f(x)", main = "convex: gradient descent works")
plot(xs, rastrigin_1d(xs), type = "l", col = BLUE, lwd = 1, xlab = "x", ylab = "f(x)", main = "rugged: 12 starts, 12 answers")
points(ends, rastrigin_1d(ends), pch = 19, col = ORANGE, cex = 1.1)
plot(1:9, acc_k, type = "s", col = BLUE, lwd = 2, xlab = "number of top-ranked features kept",
     ylab = "5-fold CV accuracy (k-NN)", main = "discrete: no gradient exists")
cat(sprintf("gradient descent on the rugged function: %d of 12 runs reached the global minimum at x = 0\n",
            sum(abs(ends) < 1e-3)))



<p class="caption"><b>Figura 1.</b> Izquierda: un objetivo convexo, donde todo camino cuesta abajo conduce al único mínimo. Centro: un corte unidimensional de la función de Rastrigin; los puntos naranjas son los lugares donde se detuvo el descenso de gradiente (L-BFGS-B) desde doce puntos de partida aleatorios —cada ejecución convergió al mínimo local más cercano, y el número de ejecuciones que alcanzaron el mínimo global en \(x = 0\) se imprime arriba. Derecha: exactitud en validación cruzada de un clasificador k-NN sobre los datos de entrenamiento de cáncer de mama, en función de cuántas de las características mejor clasificadas usa; el objetivo está definido solo en los enteros y no es monótono, así que "dar un paso cuesta abajo" carece de sentido.</p>

### 1.3 La metáfora biológica y sus límites

La idea de usar la evolución simulada para resolver problemas se desarrolló de forma independiente varias veces en la década de 1960 —las <em>estrategias evolutivas</em> de Rechenberg y Schwefel en Berlín, la <em>programación evolutiva</em> de Fogel en California—, pero el <em>algoritmo genético</em> en su forma canónica, con cromosomas binarios, selección proporcional a la aptitud, cruce y mutación, se debe a John Holland y sus estudiantes en la Universidad de Michigan. El libro de Holland de 1975 lo planteó como una teoría general de los sistemas adaptativos y aportó el <em>teorema de los esquemas</em> (Sección 6);<sup class="c"><a href="#r1">[1]</a></sup> la tesis de Kenneth De Jong de 1975 fue el primer estudio experimental sistemático del comportamiento de los GA y estableció la metodología de estudios de parámetros que sigue la Sección 3.8;<sup class="c"><a href="#r3">[3]</a></sup> y el libro de texto de David Goldberg de 1989, que contiene el ejemplo resuelto a mano de la Sección 4, hizo el método accesible a los ingenieros.<sup class="c"><a href="#r2">[2]</a></sup>

La metáfora es productiva, pero no debe tomarse al pie de la letra, y las diferencias le importan al ingeniero:

| En biología | En un algoritmo genético |
|---|---|
| La aptitud se <em>define</em> por el éxito reproductivo; no hay un objetivo externo | La aptitud es una función <em>elegida por el diseñador</em>; el algoritmo optimiza exactamente lo que la función recompensa, que a menudo no es lo que se pretendía |
| El mapa genotipo-fenotipo (el desarrollo) es enormemente complejo | La función de decodificación suele ser simple y conocida |
| Las poblaciones son enormes y el tiempo no es una restricción | Lo típico son poblaciones de 20–200 y un presupuesto de unos pocos miles de evaluaciones de aptitud, porque cada evaluación puede ser una simulación o una validación cruzada |
| La mutación y la recombinación son las únicas fuentes de novedad | El diseñador puede añadir búsqueda local, operadores de reparación y heurísticas específicas del problema |
| La evolución no tiene meta y no converge | Se espera que un GA converja, y la convergencia <em>prematura</em> (Sección 6) es su falla característica |

<div class="box warn"><h4>Error frecuente</h4><p>Un algoritmo genético no es un algoritmo de aprendizaje en el sentido de las Unidades 2–6: no produce un modelo a partir de datos, produce el <em>mejor candidato</em> para un objetivo. Cuando un GA se usa en aprendizaje automático, se usa <em>alrededor</em> de un algoritmo de aprendizaje —para elegir sus características, sus hiperparámetros o su arquitectura— y el algoritmo de aprendizaje se llama, como caja negra, dentro de la función de aptitud. La Sección 8 muestra ambos usos. La distinción que hay que retener es que el GA busca en un espacio de <em>soluciones</em>, no en un espacio de <em>hipótesis sobre los datos</em>.</p></div>

### 1.4 Dónde se ubica esta unidad

La Unidad 1 nombró cuatro enfoques de la IA. Los enfoques simbólico, conexionista y estadístico han tenido cada uno tres o cuatro unidades; el enfoque evolutivo tiene esta. Sus métodos son más antiguos que el aprendizaje profundo, más lentos que el descenso de gradiente cuando existe un gradiente, e insustituibles cuando no existe. En bioingeniería aparecen dondequiera que un espacio de diseño sea combinatorio: la selección de electrodos o de características a partir de una señal fisiológica,<sup class="c"><a href="#r33">[33]</a></sup> la disposición de los haces en la planificación de radioterapia,<sup class="c"><a href="#r29">[29]</a></sup> la secuencia de una proteína con un plegamiento prescrito,<sup class="c"><a href="#r32">[32]</a></sup> y el grafo molecular de un candidato a fármaco.<sup class="c"><a href="#r30">[30]</a></sup> Además, desde 2017, son el principal competidor de los modelos generativos profundos en el diseño molecular de novo, un punto al que vuelve la Sección 8.

## 2. Definición y vocabulario

### 2.1 Definición

Un <strong>algoritmo genético</strong> es un <em>método de búsqueda estocástico basado en poblaciones</em> que mejora iterativamente un conjunto de soluciones candidatas aplicando operadores inspirados en la selección natural y la genética —selección, recombinación (cruce) y mutación— y usando una función objetivo (la <em>aptitud</em>) como única información sobre el problema. Cada parte de la definición cumple una función:

- <em>Basado en poblaciones</em>: a diferencia del descenso de gradiente o del recocido simulado, que siguen un solo punto, un GA mantiene muchos candidatos a la vez. Esto es lo que le permite explorar varias regiones del espacio de búsqueda simultáneamente, y lo que hace posible el cruce.
- <em>Estocástico</em>: la selección, el cruce y la mutación son aleatorios. Dos ejecuciones con semillas distintas dan resultados distintos; la salida de un GA es una distribución, y la buena práctica informa la media y la dispersión sobre varias ejecuciones.
- <em>Búsqueda</em>, no <em>aprendizaje</em>: la salida es una solución candidata, no un modelo.
- <em>La aptitud como única información</em>: el algoritmo no necesita derivada, ni fórmula, ni continuidad. Necesita una función a la que pueda llamar. Esa es a la vez su generalidad y su costo: un GA que necesita 5,000 evaluaciones es barato cuando una evaluación es una fórmula y prohibitivo cuando es un experimento de una semana.

Formalmente, sea \(\Omega\) el espacio de búsqueda (cadenas binarias de longitud \(L\), vectores reales dentro de una caja, permutaciones de \(n\) elementos) y \(f: \Omega \to \mathbb{R}\) la aptitud, que se desea maximizar. Un GA mantiene una población \(P_t = \{x_1, \dots, x_N\}\) en la generación \(t\) y produce \(P_{t+1}\) mediante

<div class="eq">\[ P_{t+1} = \text{Replace}\big(P_t,\ \text{Mutate}\big(\text{Crossover}\big(\text{Select}(P_t, f)\big)\big)\big) \]
<span class="where">donde Select extrae \(N\) progenitores de \(P_t\) con una probabilidad creciente en \(f\), Crossover los empareja e intercambia partes de sus cromosomas, Mutate perturba al azar a la descendencia, y Replace decide cuáles de los individuos antiguos y nuevos sobreviven. El ciclo se detiene cuando se cumple un criterio de terminación (Sección 3.7), y se devuelve el mejor individuo visto en todo el proceso.</span></div>

### 2.2 Vocabulario: de la biología al algoritmo

Los autores de la literatura sobre GA usan vocabulario biológico, y cada término tiene un significado algorítmico preciso:

| Biología | Algoritmo | Ejemplo (selección de características, Sección 8.1) |
|---|---|---|
| **Gen** | Una posición (locus) del vector solución y el valor (alelo) que contiene | el bit 7, que vale 1 si se usa la característica 7 |
| **Cromosoma** (individuo) | La solución codificada completa, un vector de genes | una cadena binaria |
| **Genotipo** | La representación codificada sobre la que actúan los operadores | `0110100` |
| **Fenotipo** | La solución decodificada que evalúa la función de aptitud | el clasificador k-NN entrenado con las características elegidas |
| **Población** | Un multiconjunto de \(N\) cromosomas | varias máscaras de características |
| **Generación** | Una iteración del ciclo | — |
| **Aptitud** | \(f(\text{phenotype})\), el objetivo que se desea maximizar | exactitud en validación cruzada de 5 particiones |
| **Selección** | Elegir progenitores con una probabilidad creciente en la aptitud | torneo de tamaño 3 |
| **Cruce** | Producir un hijo a partir de dos progenitores combinando sus genes | cruce de un punto |
| **Mutación** | Perturbación aleatoria de los genes de un hijo | invertir cada bit con una probabilidad pequeña |
| **Elitismo** | Copiar sin cambios el o los mejores individuos a la siguiente generación | conservar los 2 mejores |

La distinción entre <em>genotipo</em> y <em>fenotipo</em> es la que más a menudo se difumina. Los operadores actúan sobre genotipos: un cruce de un punto entre dos cadenas de bits está bien definido, decodifique o no el resultado a un clasificador sensato. La aptitud es una propiedad de los fenotipos: lo que se mide es la exactitud del clasificador, no la de la cadena de bits. La función de <strong>decodificación</strong> que está entre ambos es donde se concentra la mayor parte del esfuerzo de diseño de un GA, porque una mala codificación vuelve inalcanzables las buenas soluciones o destructivos los operadores (Sección 3.1). El bloque de código siguiente muestra cada término de la tabla sobre una población de cuatro cromosomas.



In [ ]:
# Cuatro cromosomas de 5 bits que codifican enteros 0..31, aptitud f(x) = x^2 (ejemplo de Goldberg, seguido a mano en la Sección 4).
population <- rbind(c(0,1,1,0,1), c(1,1,0,0,0), c(0,1,0,0,0), c(1,0,0,1,1))
phenotype  <- decode(population)
fitness    <- phenotype^2
tab <- data.frame(
  `chromosome (genotype)` = apply(population, 1, bits2str),
  `x (phenotype)`         = phenotype,
  `fitness f(x)=x^2`      = as.integer(fitness),
  `selection prob.`       = round(fitness / sum(fitness), 3),
  `expected copies`       = round(fitness / mean(fitness), 2),
  check.names = FALSE)
cat(sprintf("population size N = %d, chromosome length L = %d, search space |Omega| = 2^L = %d\n",
            nrow(population), ncol(population), 2^ncol(population)))
cat(sprintf("sum of fitness = %.0f, mean = %.1f, max = %.0f\n", sum(fitness), mean(fitness), max(fitness)))
print(tab)



Cuatro cromosomas, cuatro fenotipos, cuatro valores de aptitud. Las dos últimas columnas anticipan la Sección 3.3: con selección proporcional a la aptitud, el segundo cromosoma, que tiene casi la mitad de la aptitud total, espera casi dos copias en la siguiente generación, y el tercero, con alrededor de un 5 %, espera una quinta parte de una copia; en la práctica, normalmente desaparecerá. Este es ya todo el mecanismo: los buenos genes se propagan, los malos desaparecen, y los operadores de la sección siguiente recombinan lo que sobrevive.



In [ ]:
# El ciclo generacional de un GA, dibujado como esquema con los gráficos base de R.
plot.new(); plot.window(xlim = c(0, 10), ylim = c(0, 3))
steps <- c("Initialise\npopulation","Evaluate\nfitness","Termination\nmet?","Select\nparents","Crossover","Mutate","Replace\n(+elitism)")
cols  <- c(BLUE, GREEN, GREY, ORANGE, ORANGE, ORANGE, BLUE)
xpos  <- c(0.3, 1.7, 3.1, 4.6, 6.0, 7.4, 8.8)
for (i in seq_along(steps)) {
  rect(xpos[i], 1.1, xpos[i] + 1.15, 2.0, col = cols[i], border = NA)
  text(xpos[i] + 0.575, 1.55, steps[i], col = "white", font = 2, cex = 0.72)
}
for (i in 1:(length(xpos) - 1)) arrows(xpos[i] + 1.15, 1.55, xpos[i + 1], 1.55, length = 0.07, lwd = 1)
segments(9.95, 1.1, 9.95, 0.5); segments(9.95, 0.5, 2.85, 0.5)
arrows(2.85, 0.5, 2.85, 1.1, length = 0.07, lwd = 1)
text(6.0, 0.62, "next generation (t = t + 1)", cex = 0.8, font = 3)
arrows(3.675, 2.0, 3.675, 2.7, length = 0.07, lwd = 1)
text(5.1, 2.75, "yes: return best individual ever seen", cex = 0.8)
text(4.0, 1.02, "no", cex = 0.8)
title(main = "One generation of a genetic algorithm")



<p class="caption"><b>Figura 2.</b> El ciclo generacional. Todo lo que está en naranja es donde ocurre la "genética" —selección, cruce, mutación— y es independiente del problema una vez fijada la codificación; todo lo que está en verde depende del problema y de nada más. Esta separación es el atractivo práctico del método: el mismo ciclo resuelve un problema de selección de características, un problema de rutas y un problema de optimización de dosis, reescribiendo solo la decodificación y la función de aptitud.</p>

## 3. Los elementos de un algoritmo genético

Todo GA sobre el que leas es una elección en cada uno de ocho puntos: (1) cómo se codifican las soluciones, (2) cómo se calcula y escala la aptitud, (3) cómo se seleccionan los progenitores, (4) cómo se recombinan, (5) cómo se muta la descendencia, (6) qué individuos sobreviven, (7) cuándo detenerse, y (8) los parámetros numéricos. Esta sección los aborda uno a uno, cada uno con un bloque de código, usando la terminología del libro de texto de Eiben y Smith, la referencia estándar moderna.<sup class="c"><a href="#r5">[5]</a></sup>

### 3.1 Codificación y decodificación

La codificación decide lo que <em>es</em> un cromosoma. Cuatro familias cubren casi todas las aplicaciones:

- **Binaria.** El GA canónico usa cadenas de bits. Cualquier cosa puede codificarse en binario: un entero por su representación en base 2, un número real \(x \in [a, b]\) por un entero de \(L\) bits, \(k\), con \(x = a + k\,(b-a)/(2^L - 1)\) —de modo que la resolución es \((b-a)/(2^L-1)\) y más bits significan una resolución más fina—, y un subconjunto por una máscara. La debilidad de la codificación binaria simple de números es el <em>acantilado de Hamming</em> (Hamming cliff): 7 = `0111` y 8 = `1000` son vecinos en valor pero difieren en los cuatro bits, así que una mutación que lleve 7 a 8 es muy improbable. Los <em>códigos Gray</em>, en los que enteros consecutivos difieren en exactamente un bit, eliminan el acantilado.
- **Entera.** Cuando los genes son naturalmente enteros con un significado (el número de neuronas de una capa, un ángulo de haz en grados, una elección categórica), codificarlos directamente evita lo artificioso de los bits.
- **De valores reales.** Para parámetros continuos, un cromosoma es simplemente un vector \(\mathbf{x} \in \mathbb{R}^d\); los operadores cambian en consecuencia, y el resultado es un <em>GA de codificación real</em> (real-coded GA), que es cercano a una estrategia evolutiva.<sup class="c"><a href="#r9">[9]</a></sup>
- **De permutación.** Cuando la solución es un ordenamiento —una ruta, un calendario—, un cromosoma es una permutación. El cruce estándar produciría elementos repetidos y faltantes, por lo que se requieren operadores especializados (cruce de orden, Sección 3.4).<sup class="c"><a href="#r10">[10]</a></sup>

El bloque de código siguiente codifica y decodifica un ejemplo de cada una.



In [ ]:
encode_real <- function(x, a, b, L) {        # x real en [a,b] -> cromosoma binario simple de L bits
  k <- round((x - a) / (b - a) * (2^L - 1)); as.integer(intToBits(k))[L:1]
}
decode_real <- function(chrom, a, b) { L <- length(chrom); a + decode(chrom) * (b - a) / (2^L - 1) }
to_gray <- function(bits) c(bits[1], bitwXor(bits[-1], bits[-length(bits)]))   # g_i = b_i XOR b_{i-1}

a <- -5.12; b <- 5.12; L <- 10
ex <- data.frame(
  encoding = c("binary (real x = 1.5, 10 bits)", "binary, Gray of 7 and 8",
               "integer (neurons per layer)", "real-valued (log10 C, log10 gamma)",
               "permutation (route through 6 cities)"),
  `chromosome (genotype)` = c(
     bits2str(encode_real(1.5, a, b, L)),
     paste(bits2str(to_gray(c(0,1,1,1))), "/", bits2str(to_gray(c(1,0,0,0)))),
     "64, 16, 4", "1.2, -2.7", "3, 0, 5, 1, 4, 2"),
  `decoded solution (phenotype)` = c(
     sprintf("x = %.4f (resolution %.4f)", decode_real(encode_real(1.5, a, b, L), a, b), (b - a)/(2^L - 1)),
     "plain 0111/1000 differ in 4 bits; Gray codes differ in 1",
     "layers of 64, 16 and 4 units",
     sprintf("C = %.1f, gamma = %.4f", 10^1.2, 10^-2.7),
     "visit city 3, then 0, 5, 1, 4, 2, return to 3"),
  check.names = FALSE)
print(ex)



La primera fila muestra el precio de la codificación binaria de un número real: con 10 bits sobre \([-5.12, 5.12]\) la resolución es de aproximadamente 0.01, y el valor 1.5 se representa por el punto más cercano de la malla, no exactamente. Cada bit extra reduce a la mitad el paso de resolución, pero duplica el espacio de búsqueda. La segunda fila muestra el acantilado de Hamming y su remedio con el código Gray. Las filas restantes no necesitan más decodificación que la interpretación, y ese es el argumento para usar la representación natural del problema siempre que exista una.<sup class="c"><a href="#r5">[5]</a></sup>

### 3.2 La función de aptitud y el escalamiento de la aptitud

La función de aptitud es el único canal a través del cual el problema le habla al algoritmo, por lo que tres reglas gobiernan su diseño. Primero, <em>debe ser una maximización</em>; un costo \(c(x)\) que se desea minimizar se convierte en una aptitud \(f(x) = -c(x)\) o \(1/(1 + c(x))\) (esta última mantiene la aptitud positiva, cosa que exige la selección proporcional a la aptitud). Segundo, <em>debe reflejar lo que se quiere</em>, incluidas las restricciones: una solución no factible se repara, se rechaza o se <em>penaliza</em> restando un término proporcional a la violación; la Sección 5.2 usa una penalización. Tercero, <em>su escala importa</em> para la selección proporcional a la aptitud, porque las probabilidades de selección son cocientes de valores de aptitud: sumar una constante a todas las aptitudes cambia las probabilidades, aunque no cambia nada del problema.

El tercer punto produce dos fallas opuestas que el libro de Goldberg trata en extenso.<sup class="c"><a href="#r2">[2]</a></sup> Al inicio de una ejecución, un individuo puede ser mucho más apto que el resto —un <em>superindividuo</em>— y apoderarse de la población antes de que se haya explorado cualquier otra cosa. Al final de una ejecución, cuando todos los individuos son casi igual de buenos, la selección proporcional a la aptitud degenera en una selección aleatoria y el progreso se estanca. Hay dos remedios estándar:

- **El escalamiento lineal** reemplaza \(f\) por \(f' = \alpha f + \beta\), con \(\alpha, \beta\) elegidos en cada generación de modo que se conserve la media y el mejor individuo reciba un número fijo de copias esperadas, \(f'_{\max} = C\, f_{\text{avg}}\), típicamente \(C = 2\) (y los valores escalados negativos se recortan a cero).
- **La aptitud basada en rangos** descarta los valores de aptitud y usa solo sus rangos: el peor individuo recibe una probabilidad de selección proporcional a \(2 - s\) y el mejor a \(s\), con \(1 \le s \le 2\), interpolando linealmente.<sup class="c"><a href="#r8">[8]</a></sup> La presión de selección es entonces constante durante toda la ejecución.

El bloque de código siguiente aplica los tres esquemas a las dos situaciones problemáticas.



In [ ]:
p_proportional <- function(f) f / sum(f)
linear_scaling <- function(f, C = 2.0) {           # Goldberg: se conserva la media, máx = C * media
  fa <- mean(f); fmax <- max(f); fmin <- min(f)
  if (fmin > (C * fa - fmax) / (C - 1)) {
    alpha <- (C - 1) * fa / (fmax - fa); beta <- fa * (fmax - C * fa) / (fmax - fa)
  } else { alpha <- fa / (fa - fmin); beta <- -fmin * fa / (fa - fmin) }
  pmax(alpha * f + beta, 0)
}
p_rank <- function(f, s = 1.8) {                   # ranking lineal: la probabilidad depende solo del rango
  N <- length(f); ranks <- rank(f, ties.method = "first") - 1
  (2 - s) / N + 2 * ranks * (s - 1) / (N * (N - 1))
}
early <- c(980, 40, 35, 30, 25, 20)                # domina un superindividuo
late  <- c(101, 100.5, 100, 99.8, 99.5, 99)        # todos casi iguales
par(mfrow = c(1, 2), mar = c(4, 4, 2.4, 1))
for (cs in list(list(early, "early run: one super-individual"), list(late, "late run: nearly equal fitness"))) {
  f <- cs[[1]]
  M <- rbind(raw = p_proportional(f), scaled = p_proportional(linear_scaling(f)), rank = p_rank(f))
  barplot(M, beside = TRUE, col = c(BLUE, ORANGE, GREEN), names.arg = sprintf("f=%g", f),
          cex.names = 0.7, ylab = "selection probability", main = cs[[2]], ylim = c(0, 1))
}
legend("topright", c("raw proportional", "linear scaling C=2", "linear ranking s=1.8"),
       fill = c(BLUE, ORANGE, GREEN), bty = "n", cex = 0.8)
for (nm in list(list("early", early), list("late", late))) {
  f <- nm[[2]]
  cat(sprintf("%s: expected copies of best  raw %.2f | scaled %.2f | rank %.2f\n", nm[[1]],
      length(f) * max(p_proportional(f)), length(f) * max(p_proportional(linear_scaling(f))), length(f) * max(p_rank(f))))
}



<p class="caption"><b>Figura 3.</b> Probabilidades de selección de seis individuos con tres transformaciones de la aptitud. Izquierda: con selección proporcional bruta, el superindividuo (aptitud 980) se lleva la mayor parte de la probabilidad y espera varias copias en una población de seis; es casi seguro que el resto desaparezca en una generación. Derecha: cuando todas las aptitudes están a menos de un 2 % unas de otras, la selección proporcional bruta le da a cada uno un sexto y la búsqueda se convierte en una caminata aleatoria. El escalamiento lineal y el ranking producen la misma presión moderada —el mejor individuo espera unas dos copias— en ambas situaciones (las copias esperadas se imprimen arriba).</p>

### 3.3 Selección

La selección crea la presión hacia mejores soluciones. Su intensidad, la <strong>presión de selección</strong>, es la perilla de ajuste central: si es demasiado débil, la población deriva sin rumbo; si es demasiado fuerte, colapsa sobre el mejor actual antes de que se encuentren regiones mejores. Cuatro esquemas dan cuenta de casi todos los GA publicados.

**La selección por ruleta (proporcional a la aptitud)** es la original de Holland. Cada individuo recibe una porción de una ruleta proporcional a su aptitud, \(p_i = f_i / \sum_j f_j\), y la ruleta se gira \(N\) veces. El número esperado de copias del individuo \(i\) es \(N p_i = f_i / \bar f\). Sus dos problemas se describieron en la Sección 3.2; un tercero es el <em>error de muestreo</em>: con solo \(N\) giros, los conteos reales pueden estar muy lejos de los esperados.

**El muestreo universal estocástico (SUS, stochastic universal sampling)**, debido a Baker, gira la ruleta una sola vez con \(N\) punteros igualmente espaciados, en lugar de \(N\) veces con un puntero, de modo que el error de muestreo se minimiza mientras los valores esperados no cambian.<sup class="c"><a href="#r8">[8]</a></sup>

**La selección por torneo** elige \(k\) individuos al azar y devuelve el mejor. No necesita escalamiento de la aptitud, funciona con aptitudes negativas o de tipo costo, y su presión se fija con un solo entero: \(k = 2\) es suave, \(k = 7\) es fuerte. Es la opción más común en la práctica moderna.<sup class="c"><a href="#r8">[8]</a></sup>

**La selección por rango** aplica la ruleta o SUS a la aptitud basada en rangos de la Sección 3.2, de modo que la presión es constante e independiente de la escala de la aptitud.

Blickle y Thiele introdujeron una forma limpia de medir la presión de selección: el <strong>tiempo de toma de control</strong> (takeover time), el número de generaciones que la selección por sí sola (sin cruce ni mutación) necesita para llenar la población de copias del único mejor individuo.<sup class="c"><a href="#r8">[8]</a></sup> El primer bloque de código dibuja la ruleta; el segundo implementa los cuatro esquemas y mide sus tiempos de toma de control.



In [ ]:
labs <- sprintf("%s (x=%d, f=%d)", apply(population, 1, bits2str), phenotype, as.integer(fitness))
pie(fitness, labels = labs, col = c(BLUE, ORANGE, GREEN, GREY),
    main = "Roulette wheel: P(select i) = f_i / sum f", cex = 0.85)



<p class="caption"><b>Figura 4.</b> La ruleta de la población inicial de Goldberg. El cromosoma 11000 ocupa alrededor de la mitad de la ruleta y 01000 alrededor del 5 %. Girar la ruleta cuatro veces da el grupo de apareamiento (mating pool); los conteos esperados son 0.58, 1.97, 0.22 y 1.23 copias (Sección 2.2), pero un conjunto particular de cuatro giros puede dar cualquier conteo; la Sección 4 sigue a mano uno de esos conjuntos.</p>



In [ ]:
select_roulette  <- function(f, n) sample.int(length(f), n, replace = TRUE, prob = f / sum(f))
select_sus <- function(f, n) {                       # un solo giro, n punteros igualmente espaciados
  cum <- cumsum(f / sum(f)); ptr <- (runif(1) + 0:(n - 1)) / n
  pmin(findInterval(ptr, cum) + 1, length(f))
}
select_tournament <- function(f, n, k = 2) {
  cand <- matrix(sample.int(length(f), n * k, replace = TRUE), ncol = k)
  cand[cbind(1:n, max.col(matrix(f[cand], ncol = k), ties.method = "first"))]
}
select_rank <- function(f, n, s = 1.8) sample.int(length(f), n, replace = TRUE, prob = p_rank(f, s))

takeover_time <- function(selector, N = 80, n_runs = 20, max_gen = 300) {
  times <- integer(n_runs); won <- logical(n_runs)
  for (r in 1:n_runs) {
    set.seed(r); f <- runif(N, 1, 2); f[1] <- 2.2; ids <- 1:N; g <- 0
    for (g in 1:max_gen) { ids <- ids[selector(f[ids], N)]; if (length(unique(ids)) == 1) break }
    times[r] <- g; won[r] <- (ids[1] == 1)
  }
  c(mean = mean(times), sd = sd(times), won = mean(won))
}
schemes <- list("roulette (proportional)" = function(f, n) select_roulette(f, n),
                "SUS (proportional)"      = function(f, n) select_sus(f, n),
                "tournament k=2" = function(f, n) select_tournament(f, n, 2),
                "tournament k=4" = function(f, n) select_tournament(f, n, 4),
                "tournament k=7" = function(f, n) select_tournament(f, n, 7),
                "linear rank s=1.8" = function(f, n) select_rank(f, n))
res <- t(sapply(schemes, takeover_time))
print(data.frame(scheme = rownames(res), `mean takeover (gen)` = round(res[, "mean"], 2),
           sd = round(res[, "sd"], 2), `fraction won by best` = round(res[, "won"], 2),
           check.names = FALSE, row.names = NULL))



La tabla ordena los esquemas según su presión. Con valores de aptitud repartidos uniformemente entre 1 y 2 y un mejor individuo solo un 10 % por encima del resto, la selección proporcional —ruleta o SUS— necesita muchas generaciones para volver uniforme la población, porque su presión depende de los <em>cocientes</em> de aptitud, que aquí son pequeños. La última columna deja en evidencia un segundo defecto de la ruleta: en una fracción de las ejecuciones la población converge a un individuo <em>distinto del mejor</em>, porque la única copia del mejor se perdió por error de muestreo en un giro temprano (deriva genética); SUS lo pierde con mucha menos frecuencia, ya que su conteo esperado siempre se redondea al menos a uno. La selección por torneo toma el control rápidamente, y aumentar \(k\) acorta aún más el tiempo de toma de control. La selección por rango con \(s = 1.8\) queda cerca de la selección por torneo con \(k = 2\), como predice la teoría.<sup class="c"><a href="#r8">[8]</a></sup> Ninguno de estos números es "el correcto": la presión adecuada depende de cuánta exploración necesite el problema (Sección 6).

<div class="box key"><h4>Ejercicio 1</h4><p>En <code>takeover_time</code>, cambia la aptitud del mejor individuo de 2.2 a 20 (un superindividuo) y vuelve a ejecutar. Ahora la selección proporcional toma el control casi de inmediato y nunca pierde al mejor individuo, mientras que la selección por torneo y por rango no cambian. Explica, a partir de las fórmulas, por qué las dos familias reaccionan de forma tan distinta a la escala de la aptitud.</p></div>

### 3.4 Cruce (recombinación)

El cruce es el operador que distingue a un GA de todos los demás métodos de búsqueda estocástica: crea un hijo <em>combinando</em> partes de dos progenitores, bajo la hipótesis de que dos buenos progenitores pueden contener partes buenas distintas y de que un hijo que herede ambas será mejor que cualquiera de ellos (la <em>hipótesis de los bloques de construcción</em>, Sección 6). Se aplica con probabilidad \(p_c\), típicamente 0.6–0.9. El operador depende de la codificación.

**Cromosomas binarios y enteros.** El cruce <em>de un punto</em> elige un sitio de corte \(c\) e intercambia las colas. Preserva los genes adyacentes, lo que constituye un sesgo oculto (<em>sesgo posicional</em>) si los genes relacionados están alejados en la cadena. El cruce <em>de dos puntos</em> intercambia el segmento entre dos sitios de corte. El cruce <em>uniforme</em> decide cada gen de forma independiente lanzando una moneda; no tiene sesgo posicional y es el más disruptivo.

**Cromosomas de valores reales.** Intercambiar coordenadas no crea nada nuevo; los operadores de codificación real generan hijos <em>entre y alrededor de</em> los progenitores. El cruce <em>aritmético</em> toma \(\mathbf{c} = \lambda \mathbf{a} + (1 - \lambda) \mathbf{b}\), de modo que la población solo puede contraerse. <em>BLX-\(\alpha\)</em> (cruce de mezcla, blend crossover) extrae cada coordenada del hijo de manera uniforme del intervalo \([\min - \alpha d,\ \max + \alpha d]\) con \(d = |a_i - b_i|\); con \(\alpha = 0.5\) la población también puede expandirse.<sup class="c"><a href="#r9">[9]</a></sup> El <em>cruce binario simulado (SBX, simulated binary crossover)</em> imita la dispersión que produce el cruce de un punto sobre números codificados en binario, con un índice de distribución \(\eta\) que controla la dispersión; es el operador por defecto en NSGA-II.<sup class="c"><a href="#r22">[22]</a></sup>

**Permutaciones.** Cualquiera de los anteriores produciría elementos repetidos y faltantes. El <em>cruce de orden (OX, order crossover)</em>, debido a Davis, copia un segmento aleatorio del progenitor \(a\) y llena el resto con los elementos del progenitor \(b\) en el orden de \(b\), saltando los elementos ya presentes. Preserva adyacencias de ambos progenitores.<sup class="c"><a href="#r10">[10]</a></sup>

El primer bloque de código muestra los tres operadores binarios; el segundo dibuja la distribución de la descendencia de los tres operadores de valores reales; el tercero sigue paso a paso el cruce de orden.



In [ ]:
one_point <- function(a, b, site = NULL) {
  L <- length(a); if (is.null(site)) site <- sample(1:(L - 1), 1)
  list(c(a[1:site], b[(site + 1):L]), c(b[1:site], a[(site + 1):L]))
}
two_point <- function(a, b, sites = NULL) {
  L <- length(a); if (is.null(sites)) sites <- sort(sample(1:(L - 1), 2))
  c1 <- sites[1]; c2 <- sites[2]; ca <- a; cb <- b
  ca[(c1 + 1):c2] <- b[(c1 + 1):c2]; cb[(c1 + 1):c2] <- a[(c1 + 1):c2]; list(ca, cb)
}
uniform_x <- function(a, b, mask = NULL) {
  if (is.null(mask)) mask <- runif(length(a)) < 0.5
  list(ifelse(mask, a, b), ifelse(mask, b, a))
}
set.seed(0); A <- rep(1, 10); B <- rep(0, 10)        # progenitores de solo unos y solo ceros hacen visible la herencia
cat(sprintf("parent a = %s   parent b = %s\n", bits2str(A), bits2str(B)))
op1 <- one_point(A, B, 4); op2 <- two_point(A, B, c(3, 7)); op3 <- uniform_x(A, B)
cat(sprintf("one-point, site 4  -> child1 = %s   child2 = %s\n", bits2str(op1[[1]]), bits2str(op1[[2]])))
cat(sprintf("two-point, sites 3,7 -> child1 = %s   child2 = %s\n", bits2str(op2[[1]]), bits2str(op2[[2]])))
cat(sprintf("uniform, random mask -> child1 = %s   child2 = %s\n", bits2str(op3[[1]]), bits2str(op3[[2]])))



Los hijos son exactamente lo que predicen las definiciones: el cruce de un punto en el sitio 4 copia los cuatro primeros genes de un progenitor y la cola del otro; el de dos puntos intercambia solo el segmento central; el uniforme mezcla gen a gen. Con estos progenitores de solo unos y solo ceros, cada bit heredado es visible.



In [ ]:
blx_alpha   <- function(a, b, alpha = 0.5) { lo <- pmin(a, b); hi <- pmax(a, b); d <- hi - lo; runif(length(a), lo - alpha * d, hi + alpha * d) }
arithmetic_x <- function(a, b) { lam <- runif(1); lam * a + (1 - lam) * b }
sbx <- function(a, b, eta = 2) {
  u <- runif(length(a))
  beta <- ifelse(u <= 0.5, (2 * u)^(1 / (eta + 1)), (1 / (2 - 2 * u))^(1 / (eta + 1)))
  sgn <- ifelse(runif(length(a)) < 0.5, 1, -1); 0.5 * ((a + b) + sgn * beta * (b - a))
}
pa <- c(1, 1); pb <- c(3, 2); set.seed(0)
par(mfrow = c(1, 3), mar = c(4, 4, 2.4, 1))
for (op in list(list(arithmetic_x, "arithmetic (on the segment)"),
                list(function(a, b) blx_alpha(a, b), "BLX-0.5 (box, +50%)"),
                list(function(a, b) sbx(a, b), "SBX, eta=2 (near parents)"))) {
  kids <- t(sapply(1:400, function(i) op[[1]](pa, pb)))
  plot(kids, pch = 19, col = adjustcolor(BLUE, 0.4), cex = 0.5, xlim = c(-0.5, 4.5), ylim = c(-0.5, 3.5),
       xlab = "x1", ylab = "x2", main = op[[2]])
  points(rbind(pa, pb), pch = 19, col = ORANGE, cex = 1.6)
}



<p class="caption"><b>Figura 5.</b> Cuatrocientos hijos de los mismos dos progenitores (naranja) con tres operadores de cruce de valores reales. El cruce aritmético coloca a cada hijo sobre el segmento entre los progenitores, de modo que una población solo puede encogerse. BLX-0.5 llena una caja que extiende la caja envolvente de los progenitores en la mitad de su ancho, de modo que los hijos pueden quedar fuera y la población puede crecer. SBX concentra cada coordenada del hijo cerca de uno u otro progenitor, con una cola que llega más allá de ellos. La elección decide si el cruce es una fuerza de contracción o de expansión, lo que fija el equilibrio entre exploración y explotación de la Sección 6.</p>



In [ ]:
order_crossover <- function(a, b, segment = NULL, verbose = FALSE) {
  n <- length(a)
  if (is.null(segment)) { ij <- sort(sample(0:(n - 1), 2)); i <- ij[1]; j <- ij[2] } else { i <- segment[1]; j <- segment[2] }
  child <- rep(-1, n); child[(i + 1):(j + 1)] <- a[(i + 1):(j + 1)]      # 1. copiar el segmento de a (índices desde 0, i..j)
  b_order <- b[(((j + 1) + 0:(n - 1)) %% n) + 1]                         # 2. b leído desde después del segmento, de forma circular
  fill <- b_order[!(b_order %in% child)]                                 #    saltando los elementos ya presentes
  num <- n - (j - i + 1)                                                 #    posiciones por llenar
  if (num > 0) { positions <- (((j + 1) + 0:(num - 1)) %% n) + 1; child[positions] <- fill }   # 3. de forma circular
  if (verbose) {
    partial <- ifelse(seq_len(n) - 1 >= i & seq_len(n) - 1 <= j, as.character(a), "_")
    cat(sprintf("parent a = %s\nparent b = %s\nstep 1: positions %d..%d copied from a -> %s\n",
                paste(a, collapse = " "), paste(b, collapse = " "), i, j, paste(partial, collapse = " ")))
    cat(sprintf("step 2: b after segment, skipping present: %s\nstep 3: child = %s\n",
                paste(fill, collapse = " "), paste(child, collapse = " ")))
  }
  child
}
set.seed(0)
pa <- c(0,1,2,3,4,5,6,7); pb <- c(7,3,0,6,1,5,2,4)
child <- order_crossover(pa, pb, segment = c(2, 4), verbose = TRUE)
cat(sprintf("valid permutation: %s\n", identical(sort(child), 0:7)))



El hijo conserva las ciudades 2, 3, 4 en las posiciones 2–4 exactamente como en el progenitor \(a\), y llena las otras posiciones con las ciudades restantes en el orden en que las visita el progenitor \(b\), comenzando después del segmento. Cada ciudad aparece exactamente una vez —la salida es una ruta válida— y el hijo hereda <em>adyacencias</em> de ambos progenitores, que es lo que debe preservar un cruce para problemas de rutas, ya que la longitud de una ruta depende solo de qué ciudades son adyacentes.<sup class="c"><a href="#r10">[10]</a></sup>

### 3.5 La mutación y la tasa de mutación

La mutación introduce genes que la población aún no contiene. Sin ella, un GA solo puede recombinar los alelos presentes en la población inicial; una vez que un alelo se pierde de todos los cromosomas, el cruce no puede recrearlo. La mutación es, por lo tanto, la garantía de <em>alcanzabilidad</em> y la base de las demostraciones de convergencia de la Sección 6.<sup class="c"><a href="#r12">[12]</a></sup>

- Mutación por **inversión de bit** (bit-flip, binaria): cada bit se invierte de forma independiente con probabilidad \(p_m\); la recomendación estándar es \(p_m \approx 1/L\), una inversión por cromosoma en promedio.<sup class="c"><a href="#r11">[11]</a></sup>
- Mutación **gaussiana** (valores reales): \(x_i \leftarrow x_i + \mathcal{N}(0, \sigma^2)\). El tamaño de paso \(\sigma\) cumple el papel de \(p_m\); las estrategias evolutivas (Sección 7) lo adaptan durante la ejecución.
- Mutación por **intercambio** (swap, permutaciones): intercambiar los elementos de dos posiciones aleatorias; el resultado es siempre válido. Las variantes son la de inserción, la de inversión y la de desorden (scramble).

La mutación es <em>disruptiva</em>: en promedio empeora al hijo. Su beneficio es estadístico —una pequeña fracción de las mutaciones son mejoras, y la selección conserva esas—, así que la tasa debe ser lo bastante alta para encontrarlas y lo bastante baja para no destruir lo que la selección ha construido. El bloque de código siguiente implementa los tres operadores y un GA binario compacto que usan los experimentos de las Secciones 3.5–3.8, y luego mide el efecto de \(p_m\) en el problema <em>OneMax</em> (maximizar el número de unos en una cadena de 50 bits, cuyo óptimo se conoce).



In [ ]:
mutate_bitflip  <- function(kids, pm) { m <- matrix(runif(length(kids)) < pm, nrow = nrow(kids)); kids[m] <- 1 - kids[m]; kids }
mutate_gaussian <- function(kids, pm, sigma, lo = -Inf, hi = Inf) {
  m <- matrix(runif(length(kids)) < pm, nrow = nrow(kids))
  pmin(pmax(kids + m * matrix(rnorm(length(kids), 0, sigma), nrow = nrow(kids)), lo), hi)
}
mutate_swap <- function(kids, pm) {
  for (i in 1:nrow(kids)) if (runif(1) < pm) { ij <- sample(ncol(kids), 2); kids[i, ij] <- kids[i, rev(ij)] }
  kids
}
binary_ga <- function(fitness, L, n_pop = 50, n_gen = 60, pc = 0.8, pm = NULL, n_elite = 1, k = 2, stall = NULL) {
  if (is.null(pm)) pm <- 1 / L
  pop <- matrix(sample(0:1, n_pop * L, replace = TRUE), nrow = n_pop); fit <- fitness(pop)
  best <- pop[which.max(fit), ]; bestf <- max(fit); hist <- matrix(NA, 0, 2); since <- 0
  for (g in 1:n_gen) {
    hist <- rbind(hist, c(max(fit), mean(fit)))
    elite <- pop[order(fit, decreasing = TRUE)[seq_len(n_elite)], , drop = FALSE]
    par <- pop[select_tournament(fit, 2 * (n_pop - n_elite), k), , drop = FALSE]
    np <- nrow(par) / 2; kids <- matrix(0, np, L)
    for (i in 1:np) { a <- par[2 * i - 1, ]; b <- par[2 * i, ]; kids[i, ] <- if (runif(1) < pc) one_point(a, b)[[1]] else a }
    pop <- if (n_elite > 0) rbind(elite, mutate_bitflip(kids, pm)) else mutate_bitflip(kids, pm)
    fit <- fitness(pop); since <- if (max(fit) > bestf) 0 else since + 1
    if (max(fit) > bestf) { best <- pop[which.max(fit), ]; bestf <- max(fit) }
    if (!is.null(stall) && since >= stall) break
  }
  hist <- rbind(hist, c(max(fit), mean(fit)))
  list(best = best, best_fitness = bestf, history = hist)
}
onemax <- function(P) rowSums(P); L <- 50
plot(NULL, xlim = c(0, 81), ylim = c(25, 51), xlab = "generation", ylab = "best fitness (mean of 8 runs)",
     main = "OneMax, L=50, N=40: the mutation rate decides whether the GA finishes")
cols <- c(GREY, BLUE, GREEN, ORANGE); leg <- c()
for (j in seq_along(list(0, 1/L, 5/L, 0.5))) {
  pm <- c(0, 1/L, 5/L, 0.5)[j]
  runs <- sapply(1:8, function(r) { set.seed(r); binary_ga(onemax, L, n_pop = 40, n_gen = 80, pm = pm)$history[, 1] })
  lines(0:80, rowMeans(runs), col = cols[j], lwd = 2)
  leg <- c(leg, sprintf("p_m=%.2f (%.0f flips); final %.1f", pm, L * pm, mean(runs[nrow(runs), ])))
}
abline(h = L, lty = 3); legend("bottomright", leg, col = cols, lwd = 2, bty = "n", cex = 0.72)



<p class="caption"><b>Figura 6.</b> Mejor aptitud por generación en OneMax (\(L = 50\), óptimo 50) para cuatro tasas de mutación, promediada sobre ocho ejecuciones. Sin mutación (gris), la población converge a los alelos con que comenzó y se queda corta del óptimo. Con \(p_m = 1/L\) (azul), el GA alcanza el óptimo o queda muy cerca de alcanzarlo. Con cinco inversiones esperadas por cromosoma (verde), el progreso es más lento, y con \(p_m = 0.5\) (naranja) cada hijo es una cadena aleatoria y el GA es una búsqueda aleatoria. Los mejores valores finales están en la leyenda.</p>

<div class="box key"><h4>Idea clave</h4><p>El cruce <em>explota</em>: combina material que la selección ya ha aprobado. La mutación <em>explora</em>: aporta material que no está ahí. Un GA con cruce y sin mutación converge a un subespacio y se queda allí; un GA con mutación y sin cruce es una población de escaladores de colinas (hill-climbers) en paralelo; los dos juntos, en las tasas adecuadas, son lo que el método es.</p></div>

### 3.6 Reemplazo y elitismo

El reemplazo decide qué individuos forman la siguiente generación. En el modelo <strong>generacional</strong>, los \(N\) descendientes reemplazan por completo a los \(N\) progenitores (el original de Holland, que se sigue a mano en la Sección 4). En el modelo de <strong>estado estacionario</strong> (steady-state), en cada paso se producen uno o dos descendientes que reemplazan al peor miembro, de modo que las generaciones se superponen. Entre ambos se encuentra el esquema \((\mu + \lambda)\) de las estrategias evolutivas (Sección 7).

El modelo generacional puro tiene un defecto bien conocido: el mejor individuo de la generación \(t\) puede no ser seleccionado, o ser destruido por el cruce o la mutación, de modo que la mejor aptitud puede <em>disminuir</em>. El <strong>elitismo</strong> —copiar sin cambios los \(e\) mejores individuos a la siguiente generación— elimina el defecto a un costo despreciable, y es una condición del teorema de convergencia de la Sección 6: Rudolph demostró que el GA canónico <em>sin</em> elitismo no converge al óptimo, mientras que la versión elitista sí lo hace.<sup class="c"><a href="#r12">[12]</a></sup> El bloque de código siguiente ejecuta el mismo GA con y sin elitismo sobre la función de Rastrigin 2-D, codificada con 12 bits por coordenada, y grafica la mejor aptitud <em>en la población actual</em>.



In [ ]:
rastrigin  <- function(X) { X <- rbind(X); 10 * ncol(X) + rowSums(X^2 - 10 * cos(2 * pi * X)) }
decode_pop <- function(P, bits = 12, lo = -5.12, hi = 5.12) {
  P <- rbind(P); w <- 2^((bits - 1):0)
  k <- cbind(P[, 1:bits, drop = FALSE] %*% w, P[, (bits + 1):(2 * bits), drop = FALSE] %*% w)
  lo + k * (hi - lo) / (2^bits - 1)
}
fit_rastrigin_bin <- function(P) -rastrigin(decode_pop(P))
L2 <- 24
plot(NULL, xlim = c(0, 60), ylim = c(1e-3, 60), log = "y", xlab = "generation",
     ylab = "cost of best individual (log)", main = "2-D Rastrigin, binary-coded (12 bits/coordinate)")
for (e in list(list(0, ORANGE, "no elitism"), list(1, BLUE, "elitism (best 1 copied)"))) {
  set.seed(3); r <- binary_ga(fit_rastrigin_bin, L2, n_pop = 40, n_gen = 60, pm = 1 / L2, n_elite = e[[1]])
  lines(0:(nrow(r$history) - 1), -r$history[, 1], col = e[[2]], lwd = 2)
  drops <- sum(diff(r$history[, 1]) < -1e-12)
  cat(sprintf("%-26s: best cost in final pop %.4f; generations where best fitness fell: %d\n", e[[3]], -r$history[nrow(r$history), 1], drops))
}
legend("topright", c("no elitism", "elitism (best 1 copied)"), col = c(ORANGE, BLUE), lwd = 2, bty = "n", cex = 0.8)



<p class="caption"><b>Figura 7.</b> Costo (aptitud negativa) del mejor individuo de la población actual, en un eje logarítmico. Sin elitismo (naranja), la curva sube repetidamente —el conteo impreso es el número de generaciones en que se perdió el mejor individuo— porque el mejor cromosoma no fue seleccionado o fue roto por el cruce o la mutación, y la ejecución tiene que redescubrirlo. Con elitismo (azul), la curva es monótona por construcción y el costo final es menor. La lección: conserva siempre la mejor solución encontrada, ya sea mediante elitismo en la población o mediante una variable separada de "mejor de todos".</p>

### 3.7 Criterios de terminación

Un GA no tiene un punto de detención natural. Se usan cuatro criterios, a menudo combinados según "lo que ocurra primero":

1. **Un número fijo de generaciones** (o de evaluaciones de aptitud, que es el presupuesto más justo al comparar algoritmos con distintos tamaños de población): el único que hace predecible el tiempo de ejecución.
2. **Se ha alcanzado una aptitud objetivo**: solo es posible cuando el óptimo, o un valor aceptable, se conoce de antemano.
3. **Estancamiento**: la mejor aptitud no ha mejorado más que una tolerancia en las últimas \(s\) generaciones. Esto adapta el tiempo de ejecución al problema, pero puede detener una ejecución que más tarde habría escapado de una meseta.
4. **Pérdida de diversidad**: la población se ha vuelto casi uniforme (Sección 6), de modo que las generaciones siguientes son en su mayoría repeticiones.

El bloque de código siguiente añade un criterio de estancamiento al experimento anterior e informa dónde se detiene la ejecución.



In [ ]:
for (stall in c(10, 15, 25)) {
  set.seed(0); r <- binary_ga(fit_rastrigin_bin, L2, n_pop = 40, n_gen = 300, pm = 1 / L2, n_elite = 1, stall = stall)
  cat(sprintf("stall = %2d generations -> stopped after %3d generations; best cost %.4f at x = (%s)\n",
      stall, nrow(r$history) - 1, -r$best_fitness, paste(round(decode_pop(matrix(r$best, nrow = 1))[1, ], 3), collapse = ", ")))
}



Con esta semilla, las ventanas de estancamiento se detienen unas pocas generaciones después de la última mejora, y las ventanas más largas solo añaden generaciones. Ese es el resultado habitual, pero no uno garantizado: con otra semilla, una ventana corta puede detenerse durante una meseta que se habría abandonado más tarde, y por eso los criterios de estancamiento se combinan con un número mínimo de generaciones. Como la codificación binaria con 12 bits por coordenada tiene una resolución de alrededor de 0.0025, el mejor costo alcanzable con esta codificación no es exactamente cero, incluso cuando el GA encuentra la celda correcta de la malla.

### 3.8 Los parámetros y sus efectos

Un GA canónico tiene cuatro parámetros numéricos —el tamaño de la población \(N\), la probabilidad de cruce \(p_c\), la probabilidad de mutación \(p_m\) y la presión de selección—, además del número de élites y del presupuesto de terminación. La tesis de De Jong recomendó \(N = 50\), \(p_c = 0.6\), \(p_m = 0.001\);<sup class="c"><a href="#r3">[3]</a></sup> Grefenstette encontró más tarde que \(N = 30\), \(p_c = 0.95\), \(p_m = 0.01\) era mejor para el rendimiento en línea (online performance); el consejo práctico que ha sobrevivido es \(p_m \approx 1/L\), \(p_c \in [0.6, 0.9]\), y \(N\) tan grande como lo permita el presupuesto de evaluaciones.<sup class="c"><a href="#r5">[5]</a></sup> Srinivas y Patnaik propusieron que \(p_c\) y \(p_m\) fueran <em>adaptativos</em> —bajos para los individuos de alta aptitud, altos para los de baja aptitud—, lo que reduce la convergencia prematura sin ajuste manual.<sup class="c"><a href="#r11">[11]</a></sup>

La interacción que más importa es la que se da entre \(N\) y \(p_m\) <em>con un presupuesto de evaluaciones fijo</em>: una población más grande explora más por generación, pero se puede permitir menos generaciones. El bloque de código siguiente fija el presupuesto en 4,000 evaluaciones de aptitud y mide el costo final en el problema de Rastrigin codificado en binario para una malla de valores de \(N\) y \(p_m\), promediado sobre cinco semillas.



In [ ]:
budget <- 4000; sizes <- c(10, 20, 50, 100); rates <- c(0, 0.5 / L2, 1 / L2, 2 / L2, 5 / L2)
grid <- matrix(0, length(sizes), length(rates))
for (i in seq_along(sizes)) for (j in seq_along(rates)) {
  costs <- sapply(1:5, function(r) { set.seed(r)
    -binary_ga(fit_rastrigin_bin, L2, n_pop = sizes[i], n_gen = budget %/% sizes[i], pm = rates[j], n_elite = 1)$best_fitness })
  grid[i, j] <- mean(costs)
}
image(seq_along(rates), seq_along(sizes), t(log10(grid + 1e-3)), col = hcl.colors(25, "YlGnBu", rev = TRUE),
      axes = FALSE, xlab = "mutation rate p_m (L = 24)", ylab = "population size N",
      main = "Mean final cost (2-D Rastrigin, 4000 evals, 5 runs)")
axis(1, seq_along(rates), c("0", "0.5/L", "1/L", "2/L", "5/L")); axis(2, seq_along(sizes), sizes)
for (i in seq_along(sizes)) for (j in seq_along(rates)) text(j, i, sprintf("%.2f", grid[i, j]), cex = 0.8)
best_ij <- which(grid == min(grid), arr.ind = TRUE)[1, ]
cat(sprintf("best combination: N = %d, p_m = %.3f (%.1f flips/chromosome) -> mean cost %.3f\n",
    sizes[best_ij[1]], rates[best_ij[2]], rates[best_ij[2]] * L2, min(grid)))



<p class="caption"><b>Figura 8.</b> Costo final medio (menor es mejor; los números son los valores y el color su logaritmo) del GA para Rastrigin codificado en binario, con un presupuesto fijo de 4,000 evaluaciones. La columna sin mutación es mala con todos los tamaños de población, porque la búsqueda queda confinada a los alelos de la población inicial. Las columnas con \(p_m = 1/L\) y \(2/L\) son en general las mejores; \(5/L\) es peor. La población más grande se queda sin generaciones; una población muy pequeña se comporta como un escalador de colinas impulsado por la mutación, lo que le conviene a este paisaje, suave a pequeña escala. Dos advertencias: el patrón es específico de este problema y de esta codificación, y cinco ejecuciones solo dan una media aproximada: vuelve a ejecutar con más semillas y algunas celdas intercambiarán posiciones.</p>

<div class="box key"><h4>Ejercicio 2</h4><p>Cambia el presupuesto a 1,000 evaluaciones y a 16,000 y vuelve a ejecutar. ¿Cómo se desplaza el mejor tamaño de población con el presupuesto? Luego reemplaza el tamaño de torneo <code>k=2</code> por <code>k=5</code> dentro de <code>binary_ga</code> y observa si una selección más fuerte ayuda más a las poblaciones grandes o a las pequeñas.</p></div>

## 4. Ejemplo resuelto a mano: el problema x² de Goldberg

El libro de texto de Goldberg comienza con una simulación a mano que todo estudiante del campo ha reproducido.<sup class="c"><a href="#r2">[2]</a></sup> El problema consiste en maximizar \(f(x) = x^2\) sobre los enteros \(x \in \{0, 1, \dots, 31\}\). La codificación es una cadena binaria de 5 bits, de modo que \(x = 31\) = `11111` es el óptimo, con \(f = 961\). La población tiene \(N = 4\), la selección es por ruleta, el cruce es de un punto con \(p_c = 1\) y la mutación es por inversión de bit con \(p_m = 0.001\). La población inicial es la que Goldberg obtuvo lanzando una moneda: 01101, 11000, 01000, 10011.

### 4.1 Paso 1 — evaluar la población

| N.º | cadena | \(x\) | \(f(x) = x^2\) | \(p_i = f_i / \sum f\) | copias esperadas \(f_i / \bar f\) |
|---|---|---|---|---|---|
| 1 | 01101 | 13 | 169 | 0.14 | 0.58 |
| 2 | 11000 | 24 | 576 | 0.49 | 1.97 |
| 3 | 01000 | 8 | 64 | 0.06 | 0.22 |
| 4 | 10011 | 19 | 361 | 0.31 | 1.23 |
| | **suma** | | **1170** | 1.00 | 4.00 |
| | **promedio** | | **293** | | |
| | **máximo** | | **576** | | |

### 4.2 Paso 2 — selección por ruleta

La ruleta tiene límites acumulados en \(0.144\), \(0.637\), \(0.692\) y \(1.000\). Para que el ejemplo sea reproducible, tomamos los cuatro giros \(u = 0.10, 0.30, 0.55, 0.80\). Cada \(u\) selecciona al primer individuo cuyo límite acumulado lo supera: \(0.10 \to\) N.º 1; \(0.30 \to\) N.º 2; \(0.55 \to\) N.º 2; \(0.80 \to\) N.º 4. Los conteos reales son, por lo tanto, 1, 2, 0, 1 —cercanos a los esperados 0.58, 1.97, 0.22, 1.23, con el individuo más débil eliminado—. El grupo de apareamiento es 01101, 11000, 11000, 10011.

### 4.3 Paso 3 — cruce de un punto

Siguiendo a Goldberg, la posición 1 del grupo se aparea con la 2 y la 3 con la 4. Los sitios de corte son el **sitio 4** para la primera pareja y el **sitio 2** para la segunda:

| pareja | progenitores (corte marcado) | hijos | \(x\) | \(f(x)\) |
|---|---|---|---|---|
| 1 × 2, sitio 4 | 0110\|1 y 1100\|0 | 01100 | 12 | 144 |
| | | 11001 | 25 | 625 |
| 3 × 4, sitio 2 | 11\|000 y 10\|011 | 11011 | 27 | 729 |
| | | 10000 | 16 | 256 |
| | | **suma** | | **1754** |
| | | **promedio** | | **439** |
| | | **máximo** | | **729** |

### 4.4 Paso 4 — mutación

Hay \(4 \times 5 = 20\) bits y cada uno se invierte con probabilidad \(0.001\), así que el número esperado de mutaciones es \(0.02\): la probabilidad de que al menos un bit se invierta es \(1 - 0.999^{20} \approx 0.02\), y en esta generación, como en la de Goldberg, no se invierte ningún bit. La nueva población es 01100, 11001, 11011, 10000.

### 4.5 Interpretación del resultado

En una generación, la aptitud promedio subió de 293 a 439 y la máxima de 576 a 729, y la población contiene ahora 11011, una cadena que no estaba presente antes y que ningún individuo podría haber producido por sí solo: combina el prefijo `11` del segundo progenitor con el sufijo `011` del cuarto. Ese es, en miniatura, el argumento de los bloques de construcción de la Sección 6. El esquema `1****`, cuyos miembros tienen todos \(x \ge 16\), pasó de dos de cuatro a tres de cuatro: los esquemas por encima del promedio ganan representación, que es lo que predice el teorema de los esquemas. El bloque de código siguiente reproduce con código cada número de las cuatro tablas, entregándole al algoritmo los mismos números aleatorios que se usaron a mano.



In [ ]:
pop0 <- rbind(c(0,1,1,0,1), c(1,1,0,0,0), c(0,1,0,0,0), c(1,0,0,1,1))
f_x2 <- function(P) decode(P)^2
f0 <- f_x2(pop0); psel <- f0 / sum(f0)
step1 <- data.frame(string = apply(pop0, 1, bits2str), x = decode(pop0), `f=x^2` = as.integer(f0),
                    p_select = round(psel, 3), `expected copies` = round(f0 / mean(f0), 2), check.names = FALSE)
cat("Step 1 - evaluation:\n"); print(step1, row.names = FALSE)
cat(sprintf("sum %.0f, average %.1f, max %.0f\n\n", sum(f0), mean(f0), max(f0)))

u <- c(0.10, 0.30, 0.55, 0.80)                          # los cuatro giros de ruleta usados a mano
chosen <- findInterval(u, cumsum(psel)) + 1
cat(sprintf("Step 2 - roulette: boundaries %s, spins %s -> individuals %s; counts %s\n",
    paste(round(cumsum(psel), 3), collapse = " "), paste(u, collapse = " "),
    paste(chosen, collapse = " "), paste(tabulate(chosen, 4), collapse = " ")))
pool <- pop0[chosen, ]

c12 <- one_point(pool[1, ], pool[2, ], site = 4); c34 <- one_point(pool[3, ], pool[4, ], site = 2)
pop1 <- rbind(c12[[1]], c12[[2]], c34[[1]], c34[[2]]); f1 <- f_x2(pop1)
step3 <- data.frame(pair = c("1x2,site4","1x2,site4","3x4,site2","3x4,site2"),
                    child = apply(pop1, 1, bits2str), x = decode(pop1), `f=x^2` = as.integer(f1), check.names = FALSE)
cat("\nStep 3 - one-point crossover:\n"); print(step3, row.names = FALSE)
cat(sprintf("sum %.0f, average %.1f, max %.0f\n", sum(f1), mean(f1), max(f1)))
set.seed(0); pm <- 0.001; flips <- matrix(runif(length(pop1)) < pm, nrow = nrow(pop1))
cat(sprintf("\nStep 4 - mutation: expected flips %.2f, P(>=1) = %.3f, flips drawn: %d\n",
    length(pop1) * pm, 1 - (1 - pm)^length(pop1), sum(flips)))
cat(sprintf("schema 1**** : %d of 4 before, %d of 4 after\n", sum(pop0[, 1] == 1), sum(pop1[, 1] == 1)))



Todas las cifras coinciden con las tablas hechas a mano: suma de 1170 a 1754, promedio de 293 a 439, máximo de 576 a 729, los hijos 01100, 11001, 11011 y 10000, ninguna mutación, y el esquema `1****` que crece de dos a tres. Confirmada una generación, el bloque de código siguiente deja que el mismo algoritmo —ahora con giros y sitios aleatorios— corra durante cincuenta generaciones con el \(p_m = 0.001\) de Goldberg y, a modo de comparación, con \(p_m = 0.05\). Con \(N = 4\), una ejecución está a merced de un puñado de números aleatorios, por lo que se muestran diez semillas para cada caso.



In [ ]:
goldberg_ga <- function(pop, n_gen, pm = 0.001) {
  hist <- matrix(NA, 0, 2)
  for (g in 1:n_gen) {
    f <- f_x2(pop); hist <- rbind(hist, c(max(f), mean(f)))
    pool <- pop[select_roulette(f, nrow(pop)), , drop = FALSE]; pool <- pool[sample(nrow(pool)), , drop = FALSE]
    kids <- matrix(0, 0, ncol(pop))
    for (i in seq(1, nrow(pool), 2)) { ch <- one_point(pool[i, ], pool[i + 1, ]); kids <- rbind(kids, ch[[1]], ch[[2]]) }
    pop <- mutate_bitflip(kids, pm)
  }
  f <- f_x2(pop); hist <- rbind(hist, c(max(f), mean(f))); list(pop = pop, history = hist)
}
par(mfrow = c(1, 2), mar = c(4, 4, 2.4, 1))
for (pc in list(list(0.001, BLUE), list(0.05, ORANGE))) {
  pm <- pc[[1]]; plot(NULL, xlim = c(0, 50), ylim = c(0, 1000), xlab = "generation",
       ylab = "best fitness in population", main = sprintf("p_m = %s  (10 seeds, N = 4)", pm))
  ever <- finals <- integer(10)
  for (sd in 0:9) { set.seed(sd); r <- goldberg_ga(pop0, 50, pm = pm)
    lines(0:50, r$history[, 1], col = adjustcolor(pc[[2]], 0.45))
    finals[sd + 1] <- max(decode(r$pop)); ever[sd + 1] <- round(sqrt(max(r$history[, 1]))) }
  abline(h = 961, lty = 3)
  cat(sprintf("p_m = %s: x=31 found at some generation in %d of 10 runs; still present at the end in %d of 10\n",
      pm, sum(ever == 31), sum(finals == 31)))
}



<p class="caption"><b>Figura 9.</b> Mejor aptitud por generación en diez ejecuciones del GA de cuatro individuos de Goldberg, con el \(p_m = 0.001\) de Goldberg (izquierda) y con \(p_m = 0.05\) (derecha); la línea punteada es el óptimo \(f(31) = 961\). Con \(p_m = 0.001\), cada ejecución converge en pocas generaciones a una población de cadenas idénticas y se queda allí, y el óptimo se encuentra en pocas ejecuciones: con cuatro individuos y casi sin mutación, un alelo que se pierde nunca se recupera. Con \(p_m = 0.05\), el óptimo se alcanza en alguna generación con mucha más frecuencia, pero sin elitismo la ruleta o una mutación pueden descartarlo de nuevo (el resumen impreso da ambos conteos). Los dos paneles muestran la convergencia prematura sin mutación y la pérdida del mejor sin elitismo, que son las razones por las que las aplicaciones reales usan poblaciones de decenas a cientos de individuos, \(p_m \approx 1/L\) y elitismo.</p>

<div class="box warn"><h4>Dónde simplifica el ejemplo del libro</h4><p>El ejemplo de Goldberg usa \(p_c = 1\) y no usa elitismo, y su población de cuatro es demasiado pequeña para que la ruleta se comporte según sus valores esperados; por eso sus conteos reales (1, 2, 0, 1) son ilustrativos más que típicos. En la formulación moderna (Eiben y Smith; el tutorial de Whitley), un GA usa \(N \ge 20\), selección por torneo o por rango, \(p_c \in [0.6, 0.9]\), \(p_m \approx 1/L\), elitismo de uno o dos individuos, e informa los resultados sobre múltiples semillas.<sup class="c"><a href="#r5">[5]</a>,<a href="#r7">[7]</a></sup> Con todo, la aritmética del ejemplo es exactamente la aritmética de todo GA; solo cambian los parámetros.</p></div>

## 5. Una implementación reutilizable, probada en tres problemas

Las Secciones 3 y 4 usaron código de propósito específico. Esta sección escribe un GA general cuyas únicas entradas son una función de aptitud y tres operadores específicos de la codificación (inicializar, cruzar, mutar). Luego se prueba en tres problemas que abarcan las tres codificaciones: una función continua con muchos óptimos locales (valores reales), el problema de la mochila 0/1 (binaria) y una instancia del problema del viajante (permutación). En cada caso, el GA se compara con la <em>búsqueda aleatoria</em> usando el mismo número de evaluaciones de aptitud, que es la línea base honesta mínima para cualquier optimizador estocástico: un método que no supera a la búsqueda aleatoria con igual presupuesto no está haciendo nada.



In [ ]:
# Un GA reutilizable: el ciclo generacional. El comportamiento específico de la codificación se entrega como funciones.
run_ga <- function(fitness, init, crossover, mutate, n_pop = 50, n_gen = 100, pc = 0.9, k = 3, n_elite = 2, snapshots = c()) {
  pop <- init(n_pop); fit <- fitness(pop); n_eval <- n_pop
  best <- pop[which.max(fit), ]; bestf <- max(fit); hist <- matrix(NA, 0, 2); snaps <- list()
  for (g in 0:(n_gen - 1)) {
    hist <- rbind(hist, c(max(fit), mean(fit)))
    if (g %in% snapshots) snaps[[as.character(g)]] <- pop
    ord <- order(fit, decreasing = TRUE); elite <- pop[ord[seq_len(n_elite)], , drop = FALSE]
    par <- pop[select_tournament(fit, 2 * (n_pop - n_elite), k), , drop = FALSE]
    np <- nrow(par) / 2; kids <- matrix(0, np, ncol(pop))
    for (i in 1:np) { a <- par[2 * i - 1, ]; b <- par[2 * i, ]; kids[i, ] <- if (runif(1) < pc) crossover(a, b) else a }
    pop <- rbind(elite, mutate(kids)); fit <- fitness(pop); n_eval <- n_eval + n_pop
    if (max(fit) > bestf) { best <- pop[which.max(fit), ]; bestf <- max(fit) }
  }
  hist <- rbind(hist, c(max(fit), mean(fit)))
  if (n_gen %in% snapshots) snaps[[as.character(n_gen)]] <- pop
  list(best = best, best_fitness = bestf, history = hist, n_eval = n_eval, snapshots = snaps)
}
# Conjuntos de operadores para las tres codificaciones: cada uno devuelve list(init, crossover, mutate).
ops_real <- function(lo, hi, d, pm = 0.2, sigma = 0.3, alpha = 0.5) list(
  init = function(n) matrix(runif(n * d, lo, hi), nrow = n),
  crossover = function(a, b) pmin(pmax(blx_alpha(a, b, alpha), lo), hi),
  mutate = function(kids) mutate_gaussian(kids, pm, sigma, lo, hi))
ops_binary <- function(L, pm = NULL) { if (is.null(pm)) pm <- 1 / L; list(
  init = function(n) matrix(sample(0:1, n * L, replace = TRUE), nrow = n),
  crossover = function(a, b) one_point(a, b)[[1]], mutate = function(kids) mutate_bitflip(kids, pm)) }
ops_permutation <- function(n_items, pm = 0.3) list(
  init = function(n) t(sapply(1:n, function(i) sample(0:(n_items - 1)))),
  crossover = function(a, b) order_crossover(a, b), mutate = function(kids) mutate_swap(kids, pm))
set.seed(0); res <- do.call(run_ga, c(list(onemax), ops_binary(50), list(n_pop = 40, n_gen = 60)))
cat(sprintf("smoke test on OneMax (L=50): best fitness %.0f of 50 after %d evaluations\n", res$best_fitness, res$n_eval))



La prueba rápida (smoke test) confirma que el ciclo general resuelve OneMax exactamente igual que el `binary_ga` de propósito específico de la Sección 3.5. De aquí en adelante, entre un problema y otro solo cambian las tres funciones de operadores y la función de aptitud.

### 5.1 Un problema de referencia continuo y multimodal: la función de Rastrigin

La función de Rastrigin, popularizada como problema de referencia para GA por Mühlenbein y colaboradores, es la prueba estándar de la capacidad de un optimizador para escapar de mínimos locales:<sup class="c"><a href="#r14">[14]</a></sup>

<div class="eq">\[ f(\mathbf{x}) = 10\,d + \sum_{i=1}^{d} \left[x_i^2 - 10 \cos(2\pi x_i)\right], \qquad x_i \in [-5.12, 5.12] \]
<span class="where">donde \(d\) es la dimensión. El término cuadrático hace que el mínimo global sea \(f(\mathbf{0}) = 0\); el término coseno añade un mínimo local cerca de cada punto entero, de modo que en dos dimensiones hay unos \(11^2 = 121\) mínimos locales dispuestos en una malla, cada uno separado de sus vecinos por una cresta de altura cercana a 20.</span></div>

El descenso de gradiente desde un punto de partida aleatorio cae en el más cercano de los 121 mínimos (Figura 1). Un GA con una población repartida por el dominio ve muchas cuencas a la vez, y la selección la concentra en la más profunda. El bloque de código siguiente ejecuta el GA de codificación real y fotografía la población en cuatro generaciones.



In [ ]:
lo <- -5.12; hi <- 5.12
set.seed(0); res_r <- do.call(run_ga, c(list(function(P) -rastrigin(P)), ops_real(lo, hi, 2),
                                        list(n_pop = 60, n_gen = 80, snapshots = c(0, 5, 15, 80))))
gx <- seq(lo, hi, length.out = 120); gy <- gx
Z <- outer(gx, gy, function(a, b) mapply(function(u, v) rastrigin(matrix(c(u, v), nrow = 1)), a, b))
par(mfrow = c(1, 4), mar = c(3.4, 3.4, 2.2, 0.6))
for (g in c(0, 5, 15, 80)) {
  image(gx, gy, Z, col = grey.colors(25, start = 0.1, end = 0.95), xlab = "x1", ylab = "x2")
  P <- res_r$snapshots[[as.character(g)]]
  points(P, pch = 21, bg = ORANGE, col = "black", cex = 0.7)
  points(0, 0, pch = 3, col = "red", lwd = 2, cex = 1.3)
  title(main = sprintf("gen %d: best cost %.3g", g, min(rastrigin(P))), cex.main = 0.95)
}
cat(sprintf("GA: best cost %.2e at x = (%s) after %d evaluations\n",
    rastrigin(matrix(res_r$best, nrow = 1)), paste(round(res_r$best, 4), collapse = ", "), res_r$n_eval))



<p class="caption"><b>Figura 10.</b> El paisaje de Rastrigin 2-D (oscuro = costo bajo; la cruz roja es el mínimo global) con la población de 60 individuos en las generaciones 0, 5, 15 y 80. La población inicial es uniforme sobre el cuadrado. En la generación 5, la selección ha vaciado las regiones de alto costo; en la generación 15 se ha concentrado en la cuenca central, con unos pocos individuos que todavía exploran las vecinas (la mutación y el cruce BLX siguen produciéndolos); en la generación 80 la población se sitúa sobre el óptimo, dentro del ancho de la mutación gaussiana. El costo final impreso está muy por debajo de la altura de las crestas, así que el GA ha encontrado la cuenca global, no simplemente una buena cuenca local.</p>

El bloque de código siguiente hace cuantitativa la comparación, sobre diez semillas y con igual presupuesto, frente a la búsqueda aleatoria pura; un algoritmo de <em>evolución diferencial</em> (el método de Storn y Price, implementado aquí desde cero y descrito en la Sección 7);<sup class="c"><a href="#r18">[18]</a></sup> y el descenso de gradiente (L-BFGS-B) reiniciado desde puntos aleatorios hasta agotar el presupuesto, que es la forma honesta de usar un optimizador local en un problema multimodal.



In [ ]:
# Evolución diferencial (rand/1/bin) desde cero, y la comparación. cost1 transforma un vector en un costo escalar.
differential_evolution <- function(cost1, lo, hi, d, NP = 20, Fw = 0.7, CR = 0.9, maxiter = 100) {
  pop <- matrix(runif(NP * d, lo, hi), nrow = NP); fit <- apply(pop, 1, cost1); nev <- NP
  for (it in 1:maxiter) for (i in 1:NP) {
    idx <- sample.int(NP, 3); while (i %in% idx) idx <- sample.int(NP, 3)
    v <- pmin(pmax(pop[idx[1], ] + Fw * (pop[idx[2], ] - pop[idx[3], ]), lo), hi)
    cr <- runif(d) < CR; cr[sample(d, 1)] <- TRUE
    trial <- ifelse(cr, v, pop[i, ]); ft <- cost1(trial); nev <- nev + 1
    if (ft < fit[i]) { pop[i, ] <- trial; fit[i] <- ft }
  }
  list(x = pop[which.min(fit), ], fun = min(fit), nfev = nev)
}
cost1 <- function(v) as.numeric(rastrigin(matrix(v, nrow = 1)))
budget <- res_r$n_eval; rows <- matrix(NA, 10, 4)
for (seed in 1:10) {
  set.seed(seed)
  ga <- do.call(run_ga, c(list(function(P) -rastrigin(P)), ops_real(lo, hi, 2), list(n_pop = 60, n_gen = 80)))
  rs <- min(rastrigin(matrix(runif(budget * 2, lo, hi), ncol = 2)))
  de <- differential_evolution(cost1, lo, hi, 2, NP = 20, maxiter = budget %/% 20 - 1)
  used <- 0; best_gd <- Inf
  while (used < budget) { r <- optim(runif(2, lo, hi), cost1, method = "L-BFGS-B", lower = lo, upper = hi)
    used <- used + r$counts[1] * 3 + 3; best_gd <- min(best_gd, r$value) }
  rows[seed, ] <- c(-ga$best_fitness, rs, de$fun, best_gd)
}
colnames(rows) <- c("genetic algorithm", "random search", "differential evolution", "multi-start gradient descent")
summ <- data.frame(`median final cost` = round(apply(rows, 2, median), 4), `worst of 10` = round(apply(rows, 2, max), 4),
                   `runs with cost < 0.01` = colSums(rows < 0.01), check.names = FALSE)
cat(sprintf("budget per method: about %d fitness evaluations\n", budget)); print(summ)



Con igual presupuesto, los dos métodos evolutivos encuentran la cuenca global en la mayoría de las diez ejecuciones cada uno, con un costo final mediano cercano a cero; las ejecuciones que fallan convergieron a una de las cuencas adyacentes al origen, cuyo costo es de alrededor de 0.995 (la columna "worst of 10"). La búsqueda aleatoria con el mismo número de muestras rara vez baja de un costo de 0.01: la probabilidad de que una muestra uniforme caiga a unas pocas centésimas del óptimo es pequeña, y la búsqueda aleatoria no tiene ningún mecanismo para concentrarse. El descenso de gradiente multiarranque es el caso interesante: cada ejecución local converge exactamente al fondo de <em>alguna</em> cuenca, así que una ejecución o bien llega a cero (si uno de sus arranques cayó en la cuenca central) o se detiene en alrededor de 1. En dos dimensiones, la cuenca central ocupa alrededor del 1 % del cuadrado, así que unos pocos cientos de arranques aleatorios bastan para dar con ella, y la ventaja de la búsqueda basada en poblaciones por unidad de esfuerzo es modesta; la Sección 7 repite la comparación en cinco dimensiones, donde el panorama cambia.<sup class="c"><a href="#r21">[21]</a></sup>

### 5.2 Un problema binario: la mochila 0/1

El <strong>problema de la mochila 0/1</strong> es el problema canónico de optimización binaria: dados \(n\) objetos con pesos \(w_i\) y valores \(v_i\) y una capacidad \(W\), elegir el subconjunto de valor total máximo cuyo peso total no exceda \(W\). En bioingeniería, la misma estructura aparece como "qué sensores desplegar dentro de un presupuesto de potencia", "qué ensayos realizar dentro de un costo", "qué características calcular dentro de una latencia". El problema es NP-difícil en general, pero para pesos enteros tiene una solución exacta por programación dinámica en tiempo \(O(nW)\), que el bloque de código siguiente usa para obtener el verdadero óptimo con fines de comprobación. El cromosoma es una máscara de \(n\) bits, y la única decisión de diseño es qué hacer con las soluciones con exceso de peso: el bloque las penaliza, de modo que el GA puede atravesar regiones no factibles pero es empujado de vuelta.<sup class="c"><a href="#r15">[15]</a></sup>



In [ ]:
set.seed(0); n_items <- 30
weights <- sample(5:39, n_items, replace = TRUE); values <- sample(10:99, n_items, replace = TRUE)
capacity <- as.integer(sum(weights) * 0.4)
knapsack_fitness <- function(P, penalty = 5) {
  w <- as.vector(P %*% weights); v <- as.vector(P %*% values)
  ifelse(w <= capacity, v, v - penalty * (w - capacity))
}
knapsack_exact <- function(w, v, W) {                 # DP sobre las capacidades: best[c] = valor máximo con peso <= c
  best <- numeric(W + 1)
  for (i in seq_along(w)) { new <- best
    if (w[i] <= W) new[(w[i] + 1):(W + 1)] <- pmax(best[(w[i] + 1):(W + 1)], best[1:(W + 1 - w[i])] + v[i])
    best <- new }
  best[W + 1]
}
optimum <- knapsack_exact(weights, values, capacity)
ga_vals <- rs_vals <- numeric(10)
for (seed in 1:10) {
  set.seed(seed); r <- do.call(run_ga, c(list(knapsack_fitness), ops_binary(n_items), list(n_pop = 80, n_gen = 100)))
  ga_vals[seed] <- if (sum(r$best * weights) <= capacity) sum(r$best * values) else NA
  set.seed(seed); R <- matrix(sample(0:1, r$n_eval * n_items, replace = TRUE), ncol = n_items)
  feas <- as.vector(R %*% weights) <= capacity; rs_vals[seed] <- max(as.vector(R %*% values)[feas])
}
cat(sprintf("%d items, capacity %d (40%% of total weight); exact optimum value = %.0f\n", n_items, capacity, optimum))
cat(sprintf("GA (80 x 100 = %d evals), 10 seeds: optimum found in %d of 10 runs (values %s)\n",
    r$n_eval, sum(ga_vals == optimum, na.rm = TRUE), paste(ga_vals, collapse = " ")))
cat(sprintf("random search, same budget: best feasible values mean %.0f (max %.0f)\n", mean(rs_vals), max(rs_vals)))
cat(sprintf("GA solution (seed 10): %d items, weight %d / %d\n", sum(r$best), sum(r$best * weights), capacity))



El GA alcanza el óptimo exacto (impreso arriba) en la mayoría de las ejecuciones o en todas, con unas 8,000 evaluaciones de aptitud frente a un espacio de búsqueda de \(2^{30} \approx 10^9\) máscaras; la búsqueda aleatoria con el mismo presupuesto queda muy por debajo del óptimo en todas las ejecuciones, porque una máscara aleatoria uniforme selecciona alrededor de la mitad de los objetos y casi siempre tiene exceso de peso. Cuando una ejecución del GA no alcanza el óptimo, típicamente termina a uno o dos objetos de distancia, en una solución de valor casi igual, sin ninguna indicación de que falte algo: un GA no ofrece ningún certificado de optimalidad. Cuando existe un método exacto con un costo aceptable, como aquí, úsalo; el dominio del GA es la inmensa mayoría de los problemas en los que no existe.

<div class="box key"><h4>Ejercicio 3</h4><p>Reduce el coeficiente de penalización de 5 a 0.5 y vuelve a ejecutar. Ahora el GA devuelve soluciones no factibles (<code>NA</code> en la lista), porque exceder la capacidad cuesta menos que el valor ganado. Luego súbelo a 50: se recupera la factibilidad, pero ¿sigue alcanzando el GA el óptimo con la misma frecuencia? Una penalización demasiado dura hace difícil acercarse a la frontera de la región factible, que es donde siempre se encuentra el óptimo de una mochila.</p></div>

### 5.3 Un problema de permutaciones: el problema del viajante

El <strong>problema del viajante (TSP, travelling-salesperson problem)</strong> busca la ruta cerrada más corta que visita \(n\) puntos una vez cada uno. Es el problema modelo de toda tarea de ordenamiento —la secuencia en que una pipeta robótica visita pocillos, un láser barre puntos, una enfermera visita pacientes, un microscopio toma imágenes de regiones— y es el problema sobre el que se desarrollaron los operadores de permutación.<sup class="c"><a href="#r10">[10]</a></sup> El bloque de código siguiente genera nueve ciudades aleatorias, calcula el óptimo exacto enumerando los \(8!/2 = 20{,}160\) recorridos distintos (factible con \(n = 9\) fijando la primera ciudad y permutando las otras ocho, \(8! = 40{,}320\) órdenes; inviable con \(n = 20\), donde serían \(6 \times 10^{16}\)) y ejecuta el GA de permutaciones con cruce de orden y mutación por intercambio.



In [ ]:
all_perms <- function(x) { if (length(x) == 1) return(matrix(x, 1))
  do.call(rbind, lapply(seq_along(x), function(i) cbind(x[i], all_perms(x[-i])))) }
set.seed(0); n_city <- 9; cities <- matrix(runif(n_city * 2, 0, 100), ncol = 2)
Dm <- as.matrix(dist(cities))
tour_length <- function(P) { P <- rbind(P) + 1; nxt <- cbind(P[, -1, drop = FALSE], P[, 1])
  rowSums(matrix(Dm[cbind(as.vector(P), as.vector(nxt))], nrow = nrow(P))) }
perms <- cbind(0, all_perms(1:(n_city - 1)))          # fijar la ciudad 0 en primer lugar
exact <- min(tour_length(perms))
set.seed(0); res_t <- do.call(run_ga, c(list(function(P) -tour_length(P)), ops_permutation(n_city), list(n_pop = 60, n_gen = 60)))
set.seed(1); R <- t(sapply(1:res_t$n_eval, function(i) sample(0:(n_city - 1)))); rs_best <- min(tour_length(R))
succ <- sum(sapply(1:6, function(sd) { set.seed(sd)
  abs(-do.call(run_ga, c(list(function(P) -tour_length(P)), ops_permutation(n_city), list(n_pop = 60, n_gen = 60)))$best_fitness - exact) < 1e-9 }))

par(mfrow = c(1, 2), mar = c(4, 4, 2.4, 1))
route <- c(res_t$best, res_t$best[1]) + 1
plot(cities, pch = 21, bg = ORANGE, col = "black", cex = 1.6, xaxt = "n", yaxt = "n", xlab = "", ylab = "",
     main = "9 cities: GA route (solid) vs best random (dashed)")
lines(cities[route, ], col = BLUE, lwd = 2)
rr <- R[which.min(tour_length(R)), ]; rr <- c(rr, rr[1]) + 1; lines(cities[rr, ], col = GREY, lwd = 1, lty = 2)
text(cities, labels = 0:(n_city - 1), pos = 3, cex = 0.8)
plot(0:60, -res_t$history[, 1], type = "l", col = BLUE, lwd = 2, ylim = range(exact, -res_t$history),
     xlab = "generation", ylab = "tour length", main = "convergence")
lines(0:60, -res_t$history[, 2], col = ORANGE, lwd = 2); abline(h = exact, lty = 3)
legend("topright", c("best in population", "population mean", "exact optimum"),
       col = c(BLUE, ORANGE, "black"), lty = c(1, 1, 3), lwd = 2, bty = "n", cex = 0.8)
cat(sprintf("GA best tour length %.2f; exact optimum %.2f; GA found optimum in %d of 6 seeds; best random tour %.2f\n",
    -res_t$best_fitness, exact, succ, rs_best))



<p class="caption"><b>Figura 11.</b> Izquierda: las nueve ciudades, la mejor ruta del GA (línea continua) y la mejor de varios miles de permutaciones aleatorias (línea discontinua); incluso el mejor recorrido aleatorio, con igual presupuesto, es más largo que la ruta óptima del GA (las dos longitudes se imprimen más abajo), y una ruta que se cruza a sí misma siempre puede acortarse deshaciendo el cruce, así que los cruces son una señal visual de suboptimalidad. Derecha: longitud del mejor recorrido y longitud media por generación; el mejor alcanza el óptimo exacto (línea punteada) en unas pocas decenas de generaciones, mientras que la media se mantiene por encima, porque la mutación por intercambio sigue perturbando a los hijos en cada generación: la población retiene diversidad en lugar de colapsar sobre el mejor recorrido. La línea impresa informa cuántas de seis semillas encontraron el óptimo exacto.</p>

<div class="box bridge"><h4>Puente con la bioinformática</h4><p>El alineamiento múltiple de secuencias, la búsqueda de árboles filogenéticos y el empaquetamiento de cadenas laterales de proteínas son todos problemas de optimización combinatoria sin un gradiente útil, y los tres tienen resolvedores basados en algoritmos genéticos en la literatura: SAGA para el alineamiento, la búsqueda de árboles basada en GA en varios paquetes de filogenética, y el trabajo de diseño de proteínas de Jones.<sup class="c"><a href="#r32">[32]</a></sup> Los métodos progresivos y heurísticos que usaste en BGM324 son más rápidos, pero son voraces; una búsqueda basada en poblaciones es la forma estándar de comprobar si la respuesta voraz puede mejorarse.</p></div>

## 6. ¿Por qué funciona? Esquemas, diversidad y convergencia prematura

### 6.1 El teorema de los esquemas

La explicación de Holland del comportamiento de los GA se construye sobre la noción de <strong>esquema</strong> (schema, plural <em>schemata</em>): una plantilla sobre el alfabeto \(\{0, 1, *\}\) en la que \(*\) significa "cualquiera". El esquema `1****` denota todas las cadenas de 5 bits que comienzan con 1 —los enteros de 16 a 31— y `1***1` denota los enteros impares mayores que 16. Un esquema \(H\) tiene un <em>orden</em> \(o(H)\), el número de posiciones fijas, y una <em>longitud definitoria</em> \(\delta(H)\), la distancia entre su primera y su última posición fija: \(o(\text{1****}) = 1\), \(\delta = 0\); \(o(\text{1***1}) = 2\), \(\delta = 4\). Toda cadena de longitud \(L\) es una instancia de \(2^L\) esquemas, así que una población de \(N\) cadenas muestrea muchos esquemas a la vez: el <em>paralelismo implícito</em> que Holland consideraba la fuente del poder del método.<sup class="c"><a href="#r1">[1]</a></sup>

El <strong>teorema de los esquemas</strong> establece cómo cambia el número \(m(H, t)\) de instancias de un esquema en una generación del GA canónico:

<div class="eq">\[ m(H, t+1) \;\ge\; m(H, t)\, \frac{f(H, t)}{\bar f(t)} \left[1 - p_c \frac{\delta(H)}{L - 1} - o(H)\, p_m \right] \]
<span class="where">donde \(f(H, t)\) es la aptitud media de las instancias de \(H\) y \(\bar f(t)\) la aptitud media de toda la población. El primer factor es la selección: un esquema más apto que el promedio gana copias en proporción. El corchete es una cota inferior de la probabilidad de que el cruce y la mutación dejen intacta una instancia: un corte de un punto cae dentro de la longitud definitoria del esquema con probabilidad \(\delta(H)/(L-1)\), y cada uno de los \(o(H)\) bits fijos sobrevive a la mutación con probabilidad \(1 - p_m\).</span></div>

El teorema dice: <em>los esquemas cortos, de orden bajo y por encima del promedio reciben un número de ensayos que crece exponencialmente en las generaciones siguientes</em>. Goldberg llamó a esos esquemas <strong>bloques de construcción</strong> (building blocks) y propuso la <strong>hipótesis de los bloques de construcción</strong>: un GA funciona descubriendo, enfatizando y recombinando bloques de construcción en soluciones completas cada vez mejores.<sup class="c"><a href="#r2">[2]</a></sup> El ejemplo a mano de la Sección 4 mostró exactamente esto: `11***` y `***11` combinándose en 11011.

### 6.2 Críticas

El teorema de los esquemas es correcto como enunciado sobre una generación, y es mucho más débil de lo que alguna vez se creyó. El tutorial de Whitley y la literatura posterior enumeran las objeciones:<sup class="c"><a href="#r7">[7]</a></sup>

- Es una <em>desigualdad</em> sobre valores esperados para <em>un</em> paso. No dice nada sobre qué esquemas están presentes después de muchas generaciones, porque \(f(H, t)\) cambia a medida que cambia la población. Iterar la cota como si \(f(H)/\bar f\) fuera constante —el origen de la palabra "exponencialmente"— no está justificado.
- Cuenta la <em>destrucción</em> de esquemas, pero ignora su <em>construcción</em>: el cruce también crea nuevas instancias a partir de progenitores que no eran instancias. La cota es, por lo tanto, holgada.
- La hipótesis de los bloques de construcción es una hipótesis, no un teorema, y es falsa para problemas <em>engañosos</em> (deceptive), aquellos en que los esquemas cortos y de orden bajo con mayor aptitud media conducen lejos del óptimo global. Peor aún, Mitchell, Forrest y Holland diseñaron las funciones <em>royal road</em> ("camino real") para que fueran ideales para el ensamblaje de bloques de construcción y encontraron que un simple escalador de colinas con mutación aleatoria superaba al GA en ellas; la explicación (el <em>autoestopismo</em>, hitchhiking) es en sí misma un fenómeno de diversidad.<sup class="c"><a href="#r4">[4]</a></sup>
- Lo que <em>sí</em> puede demostrarse rigurosamente es de otra naturaleza: Rudolph mostró, modelando el GA como una cadena de Markov, que el GA canónico con mutación pero sin elitismo <em>no</em> converge al óptimo global, mientras que el GA elitista converge con probabilidad uno cuando \(t \to \infty\).<sup class="c"><a href="#r12">[12]</a></sup> Es una garantía del tipo más débil, pero aclara para qué sirven la mutación y el elitismo.

La posición moderna aceptada es que el teorema de los esquemas es una <em>intuición</em> útil, no una teoría predictiva. El bloque de código siguiente pone a prueba el teorema en el problema de Goldberg con una población de 100: en cada generación se calcula la cota inferior a partir de la población actual y se compara con el conteo real en la siguiente, para el esquema corto `1****` y para el más largo `1***1`.



In [ ]:
schema_count <- function(P, schema) {
  chars <- strsplit(schema, "")[[1]]; fixed <- which(chars != "*")
  if (length(fixed) == 0) return(rep(TRUE, nrow(P)))
  vals <- as.integer(chars[fixed]); apply(P[, fixed, drop = FALSE], 1, function(r) all(r == vals))
}
schema_bound <- function(P, f, schema, pc, pm, L = 5) {
  inst <- schema_count(P, schema); m <- sum(inst); if (m == 0) return(0)
  pos <- which(strsplit(schema, "")[[1]] != "*"); o <- length(pos); delta <- max(pos) - min(pos)
  m * mean(f[inst]) / mean(f) * (1 - pc * delta / (L - 1) - o * pm)
}
set.seed(0); N <- 100; pc <- 0.7; pm <- 0.01; n_gen <- 12
P <- matrix(sample(0:1, N * 5, replace = TRUE), nrow = N); rec <- list(`1****` = matrix(NA, 0, 2), `1***1` = matrix(NA, 0, 2))
for (g in 1:n_gen) {
  f <- f_x2(P); bounds <- sapply(names(rec), function(h) schema_bound(P, f, h, pc, pm))
  pool <- P[select_roulette(f, N), ]; pool <- pool[sample(N), ]; kids <- matrix(0, 0, 5)
  for (i in seq(1, N, 2)) { if (runif(1) < pc) { ch <- one_point(pool[i, ], pool[i + 1, ]); kids <- rbind(kids, ch[[1]], ch[[2]]) }
                            else kids <- rbind(kids, pool[i, ], pool[i + 1, ]) }
  P <- mutate_bitflip(kids, pm)
  for (h in names(rec)) rec[[h]] <- rbind(rec[[h]], c(bounds[h], sum(schema_count(P, h))))
}
par(mfrow = c(1, 2), mar = c(4, 4, 2.4, 1))
for (h in names(rec)) { arr <- rec[[h]]
  plot(1:n_gen, arr[, 2], type = "b", pch = 19, col = BLUE, ylim = c(0, N), xlab = "generation t",
       ylab = sprintf("instances in population of %d", N),
       main = sprintf("schema %s: o=%d, delta=%d", h, length(which(strsplit(h, "")[[1]] != "*")),
                      diff(range(which(strsplit(h, "")[[1]] != "*")))))
  lines(1:n_gen, arr[, 1], type = "b", pch = 15, lty = 2, col = ORANGE)
  legend("bottomright", c("actual m(H,t+1)", "schema-theorem bound"), col = c(BLUE, ORANGE), pch = c(19, 15), lty = c(1, 2), bty = "n", cex = 0.75) }
viol <- sapply(names(rec), function(h) sum(rec[[h]][, 2] < rec[[h]][, 1]))
cat(sprintf("generations where actual count fell below the bound: %s (the bound is on the expectation, so occasional violations are sampling noise)\n",
    paste(names(viol), viol, sep = "=", collapse = ", ")))



<p class="caption"><b>Figura 12.</b> Número real de instancias de dos esquemas en la generación siguiente (azul) y la cota inferior del teorema de los esquemas calculada a partir de la generación actual (naranja), en el problema de Goldberg con \(N = 100\), \(p_c = 0.7\), \(p_m = 0.01\). Ambos esquemas están por encima del promedio y ambos se propagan rápidamente: `1****`, cuyas instancias tienen al menos cuatro veces la aptitud del resto, queda prácticamente fijado en un par de generaciones, así que para él la cota es ajustada. Para `1***1`, la cota se sitúa muy por debajo del conteo real: su longitud definitoria de 4 hace grande el término de disrupción por cruce; sin embargo, en la práctica el esquema apenas se ve alterado, porque el cruce entre instancias <em>crea</em> instancias con tanta frecuencia como las destruye. Esa brecha es, hecha visible, la segunda crítica de la Sección 6.2.</p>

### 6.3 Exploración frente a explotación

Todo método de búsqueda estocástica equilibra dos actividades. La <strong>exploración</strong> muestrea regiones sobre las que se sabe poco; la <strong>explotación</strong> refina las regiones que ya se sabe que son buenas. Con demasiada exploración, el método nunca converge (la búsqueda aleatoria es el caso límite); con demasiada explotación, converge a la primera región buena que encuentra (escalamiento de colinas voraz). En un GA, la presión de selección y el cruce explotan, la mutación y una población grande exploran. La falla característica del exceso de explotación es la <strong>convergencia prematura</strong>: la población se vuelve casi uniforme alrededor de una solución subóptima mientras la tasa de mutación es demasiado baja para escapar.

La convergencia prematura se detecta midiendo la <strong>diversidad</strong>. Dos medidas simples para una población binaria son la <em>distancia de Hamming media por pares</em> (que es igual a \(\sum_j 2 p_j (1 - p_j)\), donde \(p_j\) es la frecuencia del alelo 1 en el locus \(j\)) y el <em>número de individuos distintos</em>. El bloque de código siguiente ejecuta el GA binario para Rastrigin con una presión de selección suave (\(k = 2\)) y una fuerte (\(k = 7\)), y registra tanto el mejor costo como la diversidad.



In [ ]:
hamming_diversity <- function(P) { p1 <- colMeans(P); sum(2 * p1 * (1 - p1)) }
ga_with_diversity <- function(k, n_gen = 80, seed = 0, pm = 1 / L2, N = 40) {
  set.seed(seed); pop <- matrix(sample(0:1, N * L2, replace = TRUE), nrow = N); fit <- fit_rastrigin_bin(pop); out <- matrix(NA, 0, 3)
  for (g in 1:n_gen) {
    out <- rbind(out, c(-max(fit), hamming_diversity(pop), nrow(unique(pop))))
    elite <- pop[order(fit, decreasing = TRUE)[1], , drop = FALSE]
    par <- pop[select_tournament(fit, 2 * (N - 1), k), , drop = FALSE]; np <- nrow(par) / 2; kids <- matrix(0, np, L2)
    for (i in 1:np) { a <- par[2 * i - 1, ]; b <- par[2 * i, ]; kids[i, ] <- if (runif(1) < 0.8) one_point(a, b)[[1]] else a }
    pop <- rbind(elite, mutate_bitflip(kids, pm)); fit <- fit_rastrigin_bin(pop)
  }
  out
}
avg <- function(k) Reduce(`+`, lapply(0:7, function(sd) ga_with_diversity(k, seed = sd))) / 8
a2 <- avg(2); a7 <- avg(7)
par(mfrow = c(1, 2), mar = c(4, 4, 2.4, 1))
plot(a2[, 1], type = "l", col = BLUE, lwd = 2, log = "y", xlab = "generation", ylab = "best cost (mean of 8 runs)", main = "progress")
lines(a7[, 1], col = ORANGE, lwd = 2)
legend("topright", c(sprintf("k=2: final %.3f", a2[nrow(a2), 1]), sprintf("k=7: final %.3f", a7[nrow(a7), 1])), col = c(BLUE, ORANGE), lwd = 2, bty = "n", cex = 0.8)
plot(a2[, 2], type = "l", col = BLUE, lwd = 2, ylim = c(0, 13), xlab = "generation", ylab = "diversity", main = "diversity of the population")
lines(a7[, 2], col = ORANGE, lwd = 2); lines(a2[, 3] / 40 * 12, col = BLUE, lty = 2); lines(a7[, 3] / 40 * 12, col = ORANGE, lty = 2)
legend("topright", c("k=2 Hamming", "k=7 Hamming", "distinct (scaled)"), col = c(BLUE, ORANGE, "black"), lty = c(1, 1, 2), lwd = c(2, 2, 1), bty = "n", cex = 0.72)



<p class="caption"><b>Figura 13.</b> Izquierda: mejor costo por generación (media de ocho ejecuciones) en el problema de Rastrigin codificado en binario, con selección por torneo suave (\(k = 2\)) y fuerte (\(k = 7\)). Derecha: las dos medidas de diversidad (las líneas discontinuas son el número de individuos distintos, reescalado al eje). La selección fuerte logra un progreso inicial ligeramente más rápido —la explotación rinde al principio—, pero su diversidad colapsa en unas diez generaciones al nivel que la mutación por sí sola sostiene, y a partir de ahí la búsqueda avanza a paso de tortuga; la selección suave mantiene más distancia durante toda la ejecución y termina con un costo menor. La distancia de Hamming media de una población aleatoria de cadenas de 24 bits es 12; ambas curvas comienzan allí y descienden, pero la selección fuerte cae a unos pocos bits casi de inmediato. La diversidad debería registrarse en toda ejecución seria de un GA, exactamente como se registra la pérdida en el entrenamiento de redes neuronales (Unidad 6): una curva de diversidad plana cerca de cero significa que la ejecución ha terminado, diga lo que diga el contador de generaciones.</p>

### 6.4 Nichos y aptitud compartida

A veces la convergencia prematura no es un error que haya que retrasar, sino un comportamiento que hay que impedir: un problema multimodal puede tener varios óptimos de calidad comparable, y el usuario los quiere <em>todos</em>: planes de tratamiento alternativos, andamiajes moleculares alternativos, subconjuntos de características alternativos de igual exactitud. Un GA simple converge a uno solo. Los métodos de <strong>nichos</strong> (niching) hacen que la población se asiente en varios picos a la vez. La técnica clásica es la <strong>aptitud compartida</strong> (fitness sharing), debida a Goldberg y Richardson: la aptitud de cada individuo se divide por el número de sus vecinos dentro de un radio \(\sigma_{\text{share}}\), ponderados según su proximidad,

<div class="eq">\[ f'_i = \frac{f_i}{\sum_j \text{sh}(d_{ij})}, \qquad \text{sh}(d) = \begin{cases} 1 - d/\sigma_{\text{share}} & d < \sigma_{\text{share}} \\ 0 & \text{otherwise} \end{cases} \]
<span class="where">donde \(d_{ij}\) es la distancia entre los individuos \(i\) y \(j\). Un pico abarrotado se penaliza en proporción a su aglomeración, de modo que la población se redistribuye hasta que cada pico alberga una subpoblación proporcional a su altura.</span></div>

Sareni y Krähenbühl revisan la aptitud compartida y las demás técnicas de nichos.<sup class="c"><a href="#r13">[13]</a></sup> El bloque de código siguiente aplica la aptitud compartida a la función de cinco picos \(f(x) = \sin^6(5\pi x)\) en \([0, 1]\) que usaron Goldberg y Richardson, con y sin compartición.



In [ ]:
five_peaks <- function(x) sin(5 * pi * x)^6
niching_ga <- function(share, N = 100, n_gen = 60, sigma_share = 0.1, seed = 0) {
  set.seed(seed); x <- runif(N)
  for (g in 1:n_gen) {
    f <- five_peaks(x) + 1e-9
    if (share) { d <- abs(outer(x, x, `-`)); sh <- ifelse(d < sigma_share, 1 - d / sigma_share, 0); f <- f / rowSums(sh) }
    parents <- x[select_sus(f, N)]; parents <- parents[sample(N)]
    kids <- sapply(1:N, function(i) { a <- parents[i]; b <- parents[(i %% N) + 1]; if (runif(1) < 0.7) arithmetic_x(a, b) else a })
    x <- pmin(pmax(kids + rnorm(N, 0, 0.01) * (runif(N) < 0.3), 0), 1)
  }
  x
}
xs1 <- seq(0, 1, length.out = 500); par(mfrow = c(1, 2), mar = c(4, 4, 2.4, 1))
for (share in c(FALSE, TRUE)) {
  xe <- niching_ga(share); counts <- tabulate(pmin(pmax(round(xe * 5 - 0.5), 0), 4) + 1, 5)
  plot(xs1, five_peaks(xs1), type = "l", col = GREY, xlab = "x", ylab = "f(x) = sin^6(5 pi x)",
       main = sprintf("%s fitness sharing: %d of 5 peaks", ifelse(share, "with", "without"), sum(counts > 3)))
  points(xe, five_peaks(xe), pch = 19, col = ifelse(share, ORANGE, BLUE), cex = 0.7)
  cat(sprintf("%s sharing: individuals per peak (0..4): %s\n", ifelse(share, "with   ", "without"), paste(counts, collapse = " ")))
}



<p class="caption"><b>Figura 14.</b> Población final de 100 individuos tras 60 generaciones en una función con cinco picos iguales. Sin compartición (izquierda), la población colapsa sobre un único pico —los conteos por pico se imprimen— aunque los cinco picos son exactamente iguales; qué pico gana lo deciden los primeros sorteos aleatorios. Con aptitud compartida (derecha, \(\sigma_{\text{share}} = 0.1\)), cada pico alberga una subpoblación de tamaño comparable, porque un individuo en un pico abarrotado ve su aptitud dividida por el número de sus vecinos. El precio es que cada pico se localiza con menos precisión que la que lograría para un único pico un GA simple del mismo tamaño.</p>

<div class="box key"><h4>Idea clave</h4><p>Un GA es una máquina para convertir diversidad en aptitud. La selección consume diversidad; la mutación, las poblaciones grandes y los nichos la reponen. Toda elección de parámetros de la Sección 3 es, en definitiva, una decisión sobre la rapidez con que se gasta la diversidad, y la única forma de saber si el ritmo fue el adecuado es medirla y ejecutar más de una semilla.</p></div>

## 7. La familia ampliada: métodos evolutivos y de enjambre

El algoritmo genético es un miembro de una familia que el campo llama hoy <strong>computación evolutiva</strong>, junto con una familia vecina de métodos de <strong>inteligencia de enjambre</strong> y el más antiguo <em>recocido simulado</em>, inspirado en la física. Cada párrafo nombra un método, su idea característica y su referencia original; la monografía de Bäck ofrece el tratamiento unificado.<sup class="c"><a href="#r6">[6]</a></sup>

**Estrategias evolutivas (ES) y CMA-ES.** Desarrolladas por Rechenberg y Schwefel en Berlín en la década de 1960 para el diseño en ingeniería, las estrategias evolutivas trabajan sobre vectores de valores reales, se apoyan en la mutación gaussiana como operador principal y —su idea decisiva— <em>adaptan el tamaño de paso de la mutación durante la ejecución</em>: la regla del 1/5 de éxito de Rechenberg aumenta \(\sigma\) cuando más de un quinto de las mutaciones mejoran y lo disminuye en caso contrario. La notación \((\mu, \lambda)\) y \((\mu + \lambda)\) describe si los \(\mu\) supervivientes se eligen solo entre los \(\lambda\) hijos o entre hijos y progenitores juntos. El estado del arte moderno es la <strong>estrategia evolutiva con adaptación de la matriz de covarianza (CMA-ES)</strong> de Hansen y Ostermeier, que adapta la matriz de covarianza completa de la distribución de mutación; es la opción por defecto para la optimización continua de caja negra en hasta unos pocos cientos de dimensiones.<sup class="c"><a href="#r16">[16]</a></sup>

**Programación genética (GP).** La programación genética de Koza hace evolucionar <em>programas</em> en lugar de vectores de parámetros: los cromosomas son árboles de expresiones, el cruce intercambia subárboles y la mutación reemplaza un subárbol por uno aleatorio. Sus aplicaciones incluyen la regresión simbólica: descubrir la fórmula, no solo los coeficientes, que se ajusta a los datos.<sup class="c"><a href="#r17">[17]</a></sup> Su dificultad característica es el <em>bloat</em> (hinchazón): los árboles crecen sin límite a menos que se penalice el tamaño.

**Evolución diferencial (DE).** La evolución diferencial de Storn y Price, para problemas de valores reales, genera un candidato sumando a un miembro de la población la <em>diferencia escalada</em> de otros dos, \(\mathbf{v} = \mathbf{x}_a + F(\mathbf{x}_b - \mathbf{x}_c)\), y luego recombina y conserva el que sea mejor. Como los tamaños de paso se extraen de la propia dispersión de la población, el método se autoescala casi sin parámetros; es el método implementado desde cero en la Sección 5.1 y que se usa de nuevo más abajo.<sup class="c"><a href="#r18">[18]</a></sup>

**Optimización por enjambre de partículas (PSO).** El enjambre de partículas de Kennedy y Eberhart, inspirado en las bandadas de aves, mueve un enjambre de partículas por el espacio de búsqueda; cada partícula tiene una velocidad que es atraída hacia la mejor posición que ella misma ha encontrado y hacia la mejor que ha encontrado el enjambre. No hay selección, cruce ni mutación —la información se propaga a través del término de atracción social— y el método se usa ampliamente, incluso para la clasificación de ECG con SVM.<sup class="c"><a href="#r19">[19]</a></sup>

**Recocido simulado (SA).** El recocido simulado de Kirkpatrick, Gelatt y Vecchi es un método de un solo punto tomado de la física estadística: un vecino aleatorio se acepta si es mejor y, si es peor en \(\Delta\), con probabilidad \(\exp(-\Delta / T)\), donde la temperatura \(T\) se reduce lentamente. A temperatura alta explora, a temperatura baja explota.<sup class="c"><a href="#r20">[20]</a></sup>

**Evolución multiobjetivo.** La mayoría de los problemas de diseño tienen varios objetivos —exactitud y número de características, dosis al tumor y dosis al tejido sano, potencia y toxicidad— que no pueden reducirse a un solo número sin pesos arbitrarios. Los algoritmos evolutivos son el enfoque dominante porque una población puede representar a la vez todo el <em>frente de Pareto</em> de compromisos no dominados; NSGA-II de Deb es el algoritmo de referencia.<sup class="c"><a href="#r22">[22]</a></sup>

**El teorema de no hay almuerzo gratis.** Wolpert y Macready demostraron que, promediado sobre <em>todas las posibles</em> funciones objetivo, todo algoritmo de optimización que nunca reevalúa un punto tiene exactamente el mismo rendimiento.<sup class="c"><a href="#r21">[21]</a></sup> El teorema a menudo se malinterpreta como "todos los métodos son igual de buenos". Lo que dice es que un método solo puede superar a otros en una <em>clase</em> de problemas explotando una estructura que esa clase posee: suavidad, separabilidad, la relevancia de los bloques de construcción. El GA superó a la búsqueda aleatoria en Rastrigin, la mochila y el TSP porque esos problemas tienen estructura; el teorema garantiza la existencia de problemas en los que perdería. La consecuencia para la ingeniería: la elección del método debe justificarse por lo que se sabe del problema, no por la reputación del método.

El bloque de código siguiente implementa en R base la optimización por enjambre de partículas y un recocido simulado simple, y repite la comparación de la Sección 5.1 —GA, evolución diferencial, PSO, recocido simulado y búsqueda aleatoria— sobre la función de Rastrigin en <em>cinco</em> dimensiones, donde la cuenca global ocupa una fracción ínfima del volumen, sobre seis semillas.



In [ ]:
pso <- function(cost, lo, hi, d = 2, n_particles = 30, n_iter = 120, w = 0.7, c1 = 1.5, c2 = 1.5) {
  x <- matrix(runif(n_particles * d, lo, hi), nrow = n_particles); v <- matrix(runif(n_particles * d, -1, 1), nrow = n_particles)
  pbest <- x; pbf <- cost(x); gbest <- pbest[which.min(pbf), ]
  for (it in 1:n_iter) {
    r1 <- matrix(runif(n_particles * d), nrow = n_particles); r2 <- matrix(runif(n_particles * d), nrow = n_particles)
    v <- w * v + c1 * r1 * (pbest - x) + c2 * r2 * (matrix(gbest, n_particles, d, byrow = TRUE) - x)
    x <- pmin(pmax(x + v, lo), hi); f <- cost(x); imp <- f < pbf; pbest[imp, ] <- x[imp, ]; pbf[imp] <- f[imp]
    gbest <- pbest[which.min(pbf), ]
  }
  list(x = gbest, fun = min(pbf), nfev = n_particles * (n_iter + 1))
}
sim_anneal <- function(cost1, lo, hi, d, maxfun = 6000, T0 = 10) {
  x <- runif(d, lo, hi); fx <- cost1(x); best <- x; fbest <- fx
  for (it in 1:maxfun) { Tt <- T0 * (1 - it / maxfun)
    xn <- pmin(pmax(x + rnorm(d, 0, 0.5), lo), hi); fn <- cost1(xn)
    if (fn < fx || runif(1) < exp(-(fn - fx) / max(Tt, 1e-9))) { x <- xn; fx <- fn }
    if (fx < fbest) { best <- x; fbest <- fx } }
  list(x = best, fun = fbest, nfev = maxfun)
}
d5 <- 5; B5 <- 6000; rows5 <- matrix(NA, 6, 5)
rast5 <- function(v) as.numeric(rastrigin(matrix(v, nrow = 1)))
for (seed in 1:6) {
  set.seed(seed)
  ga <- do.call(run_ga, c(list(function(P) -rastrigin(P)), ops_real(lo, hi, d5), list(n_pop = 60, n_gen = B5 %/% 60 - 1)))
  de <- differential_evolution(rast5, lo, hi, d5, NP = 20, maxiter = B5 %/% 20 - 1)
  ps <- pso(function(P) rastrigin(P), lo, hi, d = d5, n_particles = 40, n_iter = B5 %/% 40 - 1)
  sa <- sim_anneal(rast5, lo, hi, d5, maxfun = B5)
  rs <- min(rastrigin(matrix(runif(B5 * d5, lo, hi), ncol = d5)))
  rows5[seed, ] <- c(-ga$best_fitness, de$fun, ps$fun, sa$fun, rs)
}
colnames(rows5) <- c("genetic algorithm", "differential evolution", "particle swarm", "simulated annealing", "random search")
print(data.frame(`median final cost` = round(apply(rows5, 2, median), 3), `best of 6` = round(apply(rows5, 2, min), 3),
           `worst of 6` = round(apply(rows5, 2, max), 3), `runs < 0.01` = colSums(rows5 < 0.01), check.names = FALSE))



En cinco dimensiones la clasificación cambia y se vuelve instructiva. La búsqueda aleatoria es ahora inútil, porque la cuenca central es una fracción ínfima del volumen pentadimensional. El GA, con su paso gaussiano fijo de 0.3, típicamente termina con un costo de una o dos unidades —cada unidad de costo es una coordenada atascada en la cuenca adyacente a cero— porque le falta el paso fino y adaptativo que le permitiría cruzar la última cresta una vez que la población se ha encogido; la optimización por enjambre de partículas se comporta de forma parecida. La evolución diferencial y el recocido simulado lo hacen mejor, porque los tamaños de paso de DE se extraen de la propia dispersión de la población y se reducen automáticamente a medida que la población se contrae, y el calendario de temperatura de SA cumple el mismo papel. Esta es la motivación de la autoadaptación en las estrategias evolutivas y en CMA-ES, y la razón por la que un paso de mutación fijo es lo primero que hay que cambiar en un GA de codificación real que se estanca. Nada de esto clasifica los métodos en general —el teorema de no hay almuerzo gratis prohíbe esa lectura—, pero muestra lo que cada mecanismo aporta y lo que no aporta.<sup class="c"><a href="#r21">[21]</a></sup>

<div class="box key"><h4>Ejercicio 5</h4><p>Dale al GA un paso adaptativo: en <code>ops_real</code>, reemplaza el <code>sigma</code> fijo por un valor que se multiplique por 0.95 en cada generación (mantén un contador en una clausura). Vuelve a ejecutar la comparación en cinco dimensiones. Luego prueba la regla del 1/5 de Rechenberg: aumenta <code>sigma</code> en un factor 1.2 si más de un quinto de los hijos mejoraron respecto de sus progenitores, y disminúyelo en caso contrario.</p></div>

## 8. Aplicaciones en aprendizaje automático y en medicina

### 8.1 Selección de características con un GA envoltorio (wrapper)

Elegir qué características entregarle a un clasificador es un problema de optimización binaria sobre \(2^p\) subconjuntos, y es la aplicación con la que los GA entraron en el aprendizaje automático. Siedlecki y Sklansky mostraron en 1989 que un GA con un cromosoma de \(p\) bits podía seleccionar características para un clasificador k-NN donde la búsqueda exhaustiva es imposible y la selección voraz queda atrapada;<sup class="c"><a href="#r23">[23]</a></sup> Yang y Honavar convirtieron el enfoque en estándar, con una función de aptitud que combina la exactitud del clasificador y una penalización por el número de características.<sup class="c"><a href="#r24">[24]</a></sup> En el vocabulario de Kohavi y John, este es un método <strong>envoltorio</strong> (wrapper): el algoritmo de aprendizaje se ejecuta dentro de la función de aptitud, a diferencia de un <strong>filtro</strong> (filter), que clasifica las características según un estadístico calculado sin el clasificador (la clasificación por la prueba \(F\) de la Figura 1 es un filtro).<sup class="c"><a href="#r25">[25]</a></sup> Los métodos envoltorio son más exactos y mucho más costosos, y tienen un modo de falla específico que esta sección deja en evidencia.

El experimento usa los datos de cáncer de mama de todas las unidades.<sup class="c"><a href="#r35">[35]</a></sup> La versión de R de <code>mlbench</code> tiene nueve características nucleares, menos que las treinta del conjunto de datos de la edición en Python, pero la pedagogía es idéntica: el cromosoma es una máscara de 9 bits; la aptitud es la exactitud en validación cruzada de 5 particiones de un clasificador 5-NN (estandarizado dentro de cada partición) <em>solo sobre el conjunto de entrenamiento</em>, menos una pequeña penalización por cada característica seleccionada, de modo que, entre dos máscaras de igual exactitud, se prefiere la más pequeña. Una caché asegura que una máscara ya vista no se vuelva a evaluar.



In [ ]:
p_feat <- ncol(Xc_tr)                                  # 9 características nucleares en la versión de mlbench
fs_cache <- new.env()
fs_fitness <- function(P, lam = 0.002) apply(P, 1, function(m) {
  key <- paste(m, collapse = "")
  if (is.null(fs_cache[[key]]))
    fs_cache[[key]] <- if (sum(m) == 0) 0 else cv_knn_acc(Xc_tr[, m == 1, drop = FALSE], yc_tr) - lam * sum(m)
  fs_cache[[key]]
})
set.seed(0); res_fs <- do.call(run_ga, c(list(fs_fitness), ops_binary(p_feat), list(n_pop = 20, n_gen = 20)))
mask_ga <- res_fs$best == 1
plot(0:(nrow(res_fs$history) - 1), res_fs$history[, 1], type = "l", col = BLUE, lwd = 2,
     ylim = range(res_fs$history), xlab = "generation", ylab = "CV accuracy - 0.002 x n_features",
     main = "GA feature selection: fitness per generation")
lines(0:(nrow(res_fs$history) - 1), res_fs$history[, 2], col = ORANGE, lwd = 2)
legend("bottomright", c("best fitness", "mean fitness"), col = c(BLUE, ORANGE), lwd = 2, bty = "n", cex = 0.8)
cat(sprintf("distinct masks evaluated: %d (of 2^%d = %d); best fitness %.4f\n",
    length(ls(fs_cache)), p_feat, 2^p_feat, res_fs$best_fitness))
cat(sprintf("selected %d of %d features: %s\n", sum(mask_ga), p_feat, paste(names(bc)[1:p_feat][mask_ga], collapse = ", ")))



<p class="caption"><b>Figura 15.</b> Mejor aptitud y aptitud media por generación del GA de selección de características. La mejor aptitud sube en las primeras generaciones y luego en pequeños pasos, a medida que el GA intercambia características por fracciones de punto porcentual de exactitud en validación cruzada; la media la sigue por debajo. Con solo nueve características el espacio de búsqueda es pequeño, así que el GA examina una gran fracción de él —el conteo de máscaras distintas se imprime—, pero el mecanismo es exactamente el que importa cuando \(p\) está en las decenas o centenas, donde la búsqueda exhaustiva es imposible.</p>

El subconjunto seleccionado tiene una exactitud en validación cruzada, sobre el conjunto de entrenamiento, al menos tan alta como la de todas las características. Ese número <em>no</em> es una estimación de cómo se desempeñará el subconjunto con pacientes nuevos. El GA examinó muchas máscaras y devolvió aquella cuyo puntaje de validación cruzada fue el más alto; un máximo sobre muchas estimaciones ruidosas está sesgado hacia arriba, exactamente igual que la exactitud de entrenamiento de un modelo: las particiones se usaron para <em>elegir</em>, así que ya no pueden <em>evaluar</em>. Ambroise y McLachlan documentaron este <em>sesgo de selección</em> (selection bias) en estudios de expresión génica, donde seleccionar genes sobre el conjunto de datos completo y luego hacer validación cruzada producía estimaciones de error casi perfectas para subconjuntos sin ningún valor predictivo.<sup class="c"><a href="#r26">[26]</a></sup> El remedio es un conjunto de prueba que no haya participado en la selección, o la validación cruzada anidada. El bloque de código siguiente evalúa el subconjunto del GA, el conjunto completo de características, un filtro univariado del mismo tamaño y la mejor de 500 máscaras aleatorias del mismo tamaño, cada uno sobre las particiones de entrenamiento (optimista) y sobre el conjunto de prueba intacto (honesto).



In [ ]:
test_knn_acc <- function(mask) {
  sc <- scale(Xc_tr[, mask, drop = FALSE]); ctr <- attr(sc, "scaled:center"); scl <- attr(sc, "scaled:scale"); scl[scl == 0] <- 1
  Xte <- scale(Xc_te[, mask, drop = FALSE], center = ctr, scale = scl); mean(knn(sc, Xte, yc_tr, k = 5) == yc_te)
}
k_sel <- sum(mask_ga)
mask_filter <- rep(FALSE, p_feat); mask_filter[ord_feat[1:k_sel]] <- TRUE
set.seed(0); rand_cv <- numeric(500); rand_masks <- vector("list", 500)
for (i in 1:500) { m <- sample(c(rep(TRUE, k_sel), rep(FALSE, p_feat - k_sel))); rand_masks[[i]] <- m; rand_cv[i] <- cv_knn_acc(Xc_tr[, m, drop = FALSE], yc_tr) }
mask_rand <- rand_masks[[which.max(rand_cv)]]
evaluate <- function(mask) c(cv = cv_knn_acc(Xc_tr[, mask, drop = FALSE], yc_tr), test = test_knn_acc(mask))
tab <- rbind(`all features` = evaluate(rep(TRUE, p_feat)),
             `univariate filter (top k by F)` = evaluate(mask_filter),
             `GA wrapper` = evaluate(mask_ga),
             `best of 500 random k-subsets` = evaluate(mask_rand))
fs_table <- data.frame(`CV acc (used to select)` = round(tab[, 1], 3), `test acc (held out)` = round(tab[, 2], 3),
                       `optimism (CV - test)` = round(tab[, 1] - tab[, 2], 3), check.names = FALSE)
cat(sprintf("median CV accuracy of a random %d-subset: %.3f; best of 500: %.3f; test set has %d cases\n",
    k_sel, median(rand_cv), max(rand_cv), length(yc_te)))
print(fs_table)



La tabla es la lección central de esta sección, y se lee columna por columna. En la primera columna —el puntaje de validación cruzada sobre el conjunto de entrenamiento— el subconjunto del GA tiene el valor más alto de los cuatro, como debe ser, ya que fue elegido para maximizar esa columna; el mejor subconjunto aleatorio también está por encima del subconjunto aleatorio mediano (ambos se imprimen arriba). En la segunda columna —la exactitud sobre los casos reservados que nunca se usaron— las diferencias se reducen: el subconjunto de cinco características que eligió el GA generaliza tan bien como las nueve completas, y los cuatro subconjuntos quedan dentro de unas pocas décimas de punto porcentual, lo que con 205 casos de prueba (un caso equivale a alrededor de 0.5 %) está dentro de la resolución del experimento. La tercera columna, el optimismo, es mayor para el GA envoltorio —el subconjunto seleccionado maximizando la exactitud en las particiones sobre el mayor número de candidatos— y menor para el filtro univariado, que se eligió sin consultar en absoluto las exactitudes de las particiones; en este conjunto de datos pequeño, el optimismo del conjunto completo es en sí mismo grande, un recordatorio de que las estimaciones con una sola partición son ruidosas. Sobreviven tres conclusiones: un GA envoltorio puede encontrar un conjunto de características más pequeño con poca pérdida de exactitud, lo que es valioso para la interpretabilidad y el costo de medición; su aptitud informada sobrestima su exactitud en una cantidad que crece con el número de candidatos examinados; y solo un conjunto reservado —o la validación cruzada anidada— da el número honesto.<sup class="c"><a href="#r26">[26]</a></sup>

<div class="box warn"><h4>Error frecuente</h4><p>"El GA encontró un subconjunto con un 99 % de exactitud en validación cruzada, así que nuestro modelo tiene un 99 % de exactitud". La primera cláusula es un enunciado sobre la <em>búsqueda</em>; la segunda es una afirmación sobre <em>pacientes nuevos</em>, y el puntaje de validación cruzada no puede sustentarla una vez que se ha usado para elegir entre muchos candidatos. Es la misma fuga de datos sobre la que advirtió el Suplemento S1 para el preprocesamiento, que ahora entra a través del optimizador. Todo artículo que ajusta características o hiperparámetros mediante validación cruzada e informa ese puntaje de validación cruzada como su resultado ha cometido este error.</p></div>

<div class="box key"><h4>Ejercicio 4</h4><p>Cambia la penalización <code>lam</code> de 0.002 a 0.0 y a 0.01 y vuelve a ejecutar ambos bloques de código. ¿Cuántas características conserva el GA en cada caso, y cambia la exactitud de prueba? Luego cambia la semilla del GA. Los subconjuntos seleccionados diferirán, mientras que sus exactitudes de prueba apenas lo harán: el paisaje de la selección de características tiene muchos óptimos casi equivalentes, que es lo que los métodos de nichos de la Sección 6.4 están diseñados para revelar.</p></div>

### 8.2 Ajuste de hiperparámetros

Los hiperparámetros de un algoritmo de aprendizaje —\(C\) y \(\gamma\) de una máquina de vectores de soporte RBF (Unidad 3), la profundidad de un árbol (Unidad 4), las neuronas y la tasa de aprendizaje de una red (Unidad 6)— definen un problema de optimización de caja negra: el objetivo es un puntaje de validación cruzada, no hay gradiente, las evaluaciones son costosas y el paisaje suele ser accidentado. La búsqueda en malla (grid search) es la opción por defecto; la búsqueda aleatoria suele ser más eficiente cuando solo importan unos pocos hiperparámetros; y los métodos evolutivos son la elección natural cuando el espacio es grande o mixto, como demostraron Friedrichs e Igel para las SVM con una CMA-ES.<sup class="c"><a href="#r27">[27]</a></sup> El bloque de código siguiente ajusta \(\log_{10} C \in [-2, 3]\) y \(\log_{10} \gamma \in [-4, 1]\) para una SVM-RBF sobre el conjunto de entrenamiento de cáncer de mama con el GA de codificación real, y lo compara con una malla de \(12 \times 12\) que usa aproximadamente el mismo número de validaciones cruzadas.



In [ ]:
cv_svm_acc <- function(X, y, C, gamma, folds = 5) {
  set.seed(0); fid <- sample(rep(1:folds, length.out = nrow(X)))
  mean(sapply(1:folds, function(fo) { tr <- fid != fo
    m <- svm(X[tr, ], y[tr], cost = C, gamma = gamma, scale = TRUE); mean(predict(m, X[!tr, ]) == y[!tr]) }))
}
svm_cache <- new.env()
svm_cv <- function(theta) { key <- paste(round(theta, 4), collapse = "_")
  if (is.null(svm_cache[[key]])) svm_cache[[key]] <- cv_svm_acc(Xc_tr, yc_tr, 10^theta[1], 10^theta[2])
  svm_cache[[key]] }
svm_fitness <- function(P) apply(P, 1, svm_cv)
box_lo <- c(-2, -4); box_hi <- c(3, 1)
init_svm  <- function(n) cbind(runif(n, box_lo[1], box_hi[1]), runif(n, box_lo[2], box_hi[2]))
cross_svm <- function(a, b) pmin(pmax(blx_alpha(a, b, 0.5), box_lo), box_hi)
mut_svm   <- function(kids) { k <- kids + matrix(rnorm(length(kids), 0, 0.4), nrow = nrow(kids)) * matrix(runif(length(kids)) < 0.3, nrow = nrow(kids))
  k[, 1] <- pmin(pmax(k[, 1], box_lo[1]), box_hi[1]); k[, 2] <- pmin(pmax(k[, 2], box_lo[2]), box_hi[2]); k }
set.seed(0); res_svm <- run_ga(svm_fitness, init_svm, cross_svm, mut_svm, n_pop = 12, n_gen = 12, k = 2)
gc_ax <- seq(-2, 3, length.out = 12); gg_ax <- seq(-4, 1, length.out = 12)
grid_vals <- outer(gc_ax, gg_ax, function(a, b) mapply(function(u, v) svm_cv(c(u, v)), a, b))
bg <- which(grid_vals == max(grid_vals), arr.ind = TRUE)[1, ]; best_grid <- c(gc_ax[bg[1]], gg_ax[bg[2]])
test_of <- function(th) { m <- svm(Xc_tr, yc_tr, cost = 10^th[1], gamma = 10^th[2]); mean(predict(m, Xc_te) == yc_te) }
pts <- do.call(rbind, lapply(ls(svm_cache), function(k) as.numeric(strsplit(k, "_")[[1]])))
image(gc_ax, gg_ax, grid_vals, col = hcl.colors(21, "viridis"), xlab = "log10 C", ylab = "log10 gamma",
      main = "SVM hyperparameter landscape and the GA's search")
contour(gc_ax, gg_ax, grid_vals, add = TRUE, nlevels = 8, col = "white", labcex = 0.6)
points(pts, pch = 21, bg = "white", col = "black", cex = 0.8)
points(res_svm$best[1], res_svm$best[2], pch = 8, col = ORANGE, cex = 2, lwd = 2)
points(best_grid[1], best_grid[2], pch = 22, bg = "red", cex = 1.4)
legend("bottomleft", c("GA points", "GA best", "grid best"), pch = c(21, 8, 22), pt.bg = c("white", NA, "red"),
       col = c("black", ORANGE, "black"), bty = "n", cex = 0.75)
cat(sprintf("GA  : %d CV evaluations, best CV %.4f at log10 C=%.2f, log10 gamma=%.2f -> test accuracy %.3f\n",
    nrow(pts), res_svm$best_fitness, res_svm$best[1], res_svm$best[2], test_of(res_svm$best)))
cat(sprintf("grid: %d CV evaluations, best CV %.4f at log10 C=%.2f, log10 gamma=%.2f -> test accuracy %.3f\n",
    length(grid_vals), max(grid_vals), best_grid[1], best_grid[2], test_of(best_grid)))



<p class="caption"><b>Figura 16.</b> La exactitud en validación cruzada de la SVM-RBF sobre el plano \((\log_{10} C, \log_{10} \gamma)\), obtenida de la malla de \(12 \times 12\), con los puntos que evaluó el GA (blanco), el mejor del GA (estrella) y el mejor de la malla (cuadrado). El paisaje tiene una amplia cresta de exactitud casi igual —un \(C\) grande compensa un \(\gamma\) pequeño— y una región en \(\gamma\) grande donde el kernel se vuelve tan estrecho que cada caso de entrenamiento es su propia isla (Unidad 3). Los puntos del GA se agrupan a lo largo de la cresta después de las primeras generaciones; ambos métodos alcanzan la misma exactitud en validación cruzada y exactitudes en prueba que difieren a lo sumo en un caso de prueba (impresas arriba).</p>

En un paisaje bidimensional y suave como este, la búsqueda en malla es perfectamente adecuada y el GA no tiene ninguna ventaja; la comparación se muestra para hacer visible el mecanismo, no para recomendar el GA en este caso. Las ventajas del GA aparecen cuando el espacio tiene cinco o diez dimensiones (donde una malla de 12 puntos por eje es imposible), cuando mezcla elecciones continuas y categóricas (tipo de kernel, preprocesamiento, subconjunto de características; Huang y Wang ajustaron los parámetros de la SVM y su subconjunto de características en un solo cromosoma) y cuando el presupuesto es pequeño. La advertencia de la Sección 8.1 se aplica sin cambios: el mejor puntaje de validación cruzada encontrado por cualquiera de los dos métodos es optimista, y la exactitud en prueba es el número honesto.

### 8.3 Neuroevolución

La evolución también puede reemplazar o complementar al descenso de gradiente en el entrenamiento de redes neuronales. En la <strong>neuroevolución</strong>, el cromosoma codifica los pesos de una red de arquitectura fija (un GA de valores reales con la pérdida como aptitud negativa, útil cuando la pérdida no es diferenciable, como en el aprendizaje por refuerzo con recompensas escasas) o codifica la <em>arquitectura misma</em>. NEAT, de Stanley y Miikkulainen, hace evolucionar tanto los pesos como la topología, partiendo de redes mínimas y añadiendo neuronas y conexiones mediante mutaciones estructurales, con un esquema de marcas históricas que da sentido al cruce entre topologías distintas y un mecanismo de especiación (nichos) que protege las nuevas estructuras.<sup class="c"><a href="#r28">[28]</a></sup> El descendiente moderno es la <em>búsqueda de arquitecturas neuronales</em> (neural architecture search), en la que los algoritmos evolutivos compiten con el aprendizaje por refuerzo y con los métodos basados en gradiente para diseñar arquitecturas de redes profundas; las redes de la Unidad 8 son producto de tales búsquedas con más frecuencia que del diseño manual.

### 8.4 Aplicaciones biomédicas

El enfoque evolutivo aparece en todas las áreas de la bioingeniería médica donde un espacio de diseño es combinatorio y el objetivo es una simulación o un resultado medido. Cinco ejemplos documentados:

**Planificación de tratamientos de radioterapia.** Un plan especifica las direcciones, formas y pesos de los haces para entregar una dosis prescrita al tumor manteniendo la dosis a los órganos de riesgo por debajo de la tolerancia: un problema con restricciones, multiobjetivo y no convexo. Ezzell aplicó un GA a la optimización geométrica y de pesos de planes conformacionales tridimensionales y encontró planes competitivos con los de planificadores experimentados;<sup class="c"><a href="#r29">[29]</a></sup> hoy los métodos evolutivos multiobjetivo de la familia NSGA-II presentan a los planificadores el frente de compromisos en lugar de un único plan.<sup class="c"><a href="#r22">[22]</a></sup>

**Diseño molecular de novo.** Se estima que el espacio de diseño de las moléculas de tipo fármaco abarca \(10^{60}\) estructuras, y el objetivo —unión, solubilidad, sintetizabilidad— se calcula mediante modelos. El GA basado en grafos de Jensen muta y recombina directamente grafos moleculares, e igualó o superó a los modelos generativos profundos en tareas estándar de optimización de propiedades con una fracción del costo;<sup class="c"><a href="#r30">[30]</a></sup> en el conjunto de pruebas GuacaMol, un algoritmo genético basado en SMILES estuvo entre las líneas base más fuertes, por delante de varios generadores neuronales.<sup class="c"><a href="#r31">[31]</a></sup> Esta es la aplicación más cercana al trabajo de QSAR de BGM324: la función de aptitud del GA es un modelo QSAR.

**Diseño de proteínas.** Jones usó un GA para buscar en el espacio de secuencias aquellas compatibles con un plegamiento de esqueleto dado, puntuadas mediante potenciales de pares de residuos, en uno de los primeros estudios computacionales de diseño de proteínas de novo;<sup class="c"><a href="#r32">[32]</a></sup> la búsqueda evolutiva sobre secuencias sigue siendo un componente de los pipelines de diseño que hoy usan funciones de energía basadas en aprendizaje profundo.

**Señales fisiológicas.** La selección de características y parámetros para la clasificación de arritmias en el ECG es una instancia directa de la Sección 8.1: Nasiri y colaboradores combinaron un GA con una SVM para seleccionar las características y ajustar el clasificador para la clasificación de latidos en la base de datos MIT-BIH,<sup class="c"><a href="#r33">[33]</a></sup> y Melgani y Bazi hicieron lo mismo con optimización por enjambre de partículas;<sup class="c"><a href="#r19">[19]</a></sup> el mismo patrón se repite en la selección de canales de EEG y en el diseño de bancos de filtros.

**Selección de genes en datos ómicos.** Con miles de genes y decenas de pacientes, seleccionar una firma génica pequeña es el caso extremo del problema de envoltorio. El método GA/k-NN de Li y colaboradores seleccionó subconjuntos de genes para la clasificación de tumores y estudió la sensibilidad del resultado a los parámetros del GA, encontrando muchos subconjuntos casi equivalentes;<sup class="c"><a href="#r34">[34]</a></sup> el análisis del sesgo de selección de Ambroise y McLachlan, que la Sección 8.1 reprodujo en miniatura, se escribió precisamente sobre estos estudios y sigue siendo lectura obligatoria antes de cualquier selección de características en datos ómicos.<sup class="c"><a href="#r26">[26]</a></sup>

<div class="box bridge"><h4>Puente con la Unidad 9 y con el laboratorio</h4><p>El Laboratorio 5 aplica un GA propio a uno de estos tipos de problema; la implementación de la Sección 5 es un punto de partida, y las comprobaciones de las Secciones 6.3 (registrar la diversidad, ejecutar varias semillas) y 8.1 (reservar un conjunto de prueba) son los criterios con los que se evaluará. La Unidad 9 vuelve sobre la dimensión ética: a un optimizador le es indiferente lo que optimiza, y una función de aptitud que codifique una etiqueta inequitativa se optimizará con la misma eficiencia que una justa.</p></div>

## 9. Resumen

Ahora deberías ser capaz de explicar, sin apuntes:

- Por qué la optimización es un pilar de la IA por derecho propio, y las cuatro propiedades de un objetivo —discreto, no diferenciable, accidentado, de caja negra— bajo las cuales los métodos de gradiente fallan y la búsqueda basada en poblaciones es apropiada.
- La definición de un algoritmo genético como una búsqueda estocástica basada en poblaciones guiada por una función de aptitud, y el significado de gen, cromosoma, genotipo, fenotipo, población, generación, aptitud, selección, cruce, mutación y elitismo.
- Las ocho decisiones de diseño de un GA, con al menos dos alternativas para cada una: codificación (binaria, Gray, entera, real, de permutación); la aptitud y su escalamiento (bruta, lineal, por rango); la selección (ruleta, SUS, torneo, rango) y cómo medir su presión (tiempo de toma de control); el cruce (de un punto, de dos puntos, uniforme; aritmético, BLX-\(\alpha\), SBX; cruce de orden); la mutación (inversión de bit con \(\approx 1/L\), gaussiana, intercambio); el reemplazo (generacional, de estado estacionario) y el elitismo; la terminación; y los parámetros y su interacción con un presupuesto fijo.
- El ejemplo \(x^2\) de Goldberg: cómo construir la tabla de aptitudes, calcular los conteos esperados, girar la ruleta, aplicar el cruce de un punto en sitios dados y comprobar si hay mutación, y por qué una generación elevó la aptitud media de 293 a 439.
- Cómo escribir un ciclo de GA general cuyas partes específicas del problema son la función de aptitud y tres operadores, y cómo aplicarlo a un problema continuo, a uno binario y a uno de permutaciones; por qué la búsqueda aleatoria con igual presupuesto es la línea base mínima.
- El teorema de los esquemas —su enunciado, el significado del orden y de la longitud definitoria, la hipótesis de los bloques de construcción— y sus limitaciones; y lo que el resultado de convergencia de Rudolph garantiza y lo que no.
- Exploración frente a explotación, convergencia prematura, cómo se mide la diversidad y por qué debe registrarse, y cómo la aptitud compartida mantiene varios nichos.
- Un párrafo sobre cada uno de los siguientes: estrategias evolutivas y CMA-ES, programación genética, evolución diferencial, optimización por enjambre de partículas, recocido simulado y evolución multiobjetivo, y lo que realmente enuncia el teorema de no hay almuerzo gratis.
- Cómo un GA envoltorio selecciona características y ajusta hiperparámetros, por qué su aptitud informada es una estimación con sesgo optimista, y cómo un conjunto de prueba reservado o una validación cruzada anidada dan el número honesto.
- Cinco aplicaciones biomédicas documentadas: planificación de radioterapia, diseño molecular de novo, diseño de proteínas, selección de características de ECG y selección de firmas génicas.

## 10. Autoevaluación

<details><summary>1. Un colega quiere usar el descenso de gradiente para elegir qué 10 de 40 biomarcadores incluir en un panel diagnóstico. Da dos razones por las que esto no puede funcionar y nombra un método apropiado.</summary>El espacio de búsqueda es discreto (máscaras binarias), así que no hay derivada con respecto a la inclusión; y el objetivo (la exactitud en validación cruzada) es constante por tramos, así que incluso una versión continua relajada tiene gradiente cero casi en todas partes. Es apropiado un GA envoltorio con un cromosoma de 40 bits (Sección 8.1), u otra búsqueda combinatoria, con un conjunto de prueba reservado para la estimación final.</details>

<details><summary>2. Distingue genotipo y fenotipo en un GA que ajusta una SVM, e indica sobre cuál actúa cada operador y la función de aptitud.</summary>El genotipo es el vector \((\log_{10} C, \log_{10} \gamma)\); el fenotipo es la SVM entrenada con esos hiperparámetros. El cruce y la mutación actúan sobre el genotipo (producen nuevos vectores); la función de aptitud decodifica el genotipo en un fenotipo (ajusta la SVM) y lo mide (exactitud en validación cruzada).</details>

<details><summary>3. Calcula a mano: una población tiene valores de aptitud 10, 30, 40, 120. ¿Cuál es la probabilidad de selección de cada uno con selección por ruleta, y el número esperado de copias en un grupo de apareamiento de cuatro?</summary>La suma es 200, así que las probabilidades son 0.05, 0.15, 0.20, 0.60. La aptitud media es 50, así que las copias esperadas \(f_i/\bar f\) son 0.2, 0.6, 0.8, 2.4 (que suman 4). El último individuo espera ocupar el 60 % del grupo en una generación: un superindividuo que el escalamiento lineal o la selección por rango moderarían.</details>

<details><summary>4. En el ejemplo de Goldberg, los progenitores 11000 y 10011 cruzados en el sitio 2 dieron 11011 y 10000. ¿Qué daría el cruce en el sitio 3, y qué sitio produciría 11111 a partir de estos dos progenitores?</summary>Sitio 3: 110|00 y 100|11 dan 11011 y 10000, los mismos hijos, porque el bit 3 de ambos progenitores es 0. Ningún sitio produce 11111: los progenitores comparten un 0 en la posición 3, y el cruce de un punto no puede crear un alelo que ninguno de los progenitores porte. Solo la mutación puede aportar el 1 que falta (Sección 3.5).</details>

<details><summary>5. Lee el código: en <code>binary_ga</code>, ¿qué le ocurre al mejor individuo si <code>n_elite=0</code>, y qué figura muestra la consecuencia?</summary>Con <code>n_elite=0</code>, toda la generación siguiente está formada por descendientes mutados de progenitores seleccionados por torneo; el mejor sobrevive solo si es seleccionado y luego no se cruza (probabilidad \(1 - p_c\)) ni se muta, así que con frecuencia se pierde y la mejor aptitud puede caer entre generaciones. La Figura 7 lo muestra, y el teorema de Rudolph dice que el GA no elitista no converge al óptimo.</details>

<details><summary>6. ¿Por qué una tasa de mutación de \(1/L\) es un valor por defecto sensato, y qué ocurre con \(p_m = 0.5\)?</summary>Con \(p_m = 1/L\), cada hijo difiere de su progenitor en un bit en promedio, así que la mayoría de los hijos son variaciones pequeñas que la selección puede evaluar de forma significativa, mientras que todo alelo sigue siendo alcanzable. Con \(p_m = 0.5\), cada bit se fija al azar independientemente del progenitor, así que los hijos no llevan información de sus progenitores y el algoritmo degenera en una búsqueda aleatoria (Figura 6).</details>

<details><summary>7. El teorema de los esquemas dice que los esquemas cortos, de orden bajo y por encima del promedio reciben un número de ensayos "que crece exponencialmente". ¿Por qué la palabra "exponencialmente" no está justificada por el teorema tal como se enuncia?</summary>El teorema acota \(m(H, t+1)\) en términos de \(m(H, t)\) y del cociente \(f(H, t)/\bar f(t)\) <em>en la generación \(t\)</em>. A medida que el esquema se propaga, sus instancias forman una parte mayor de la población y el cociente cae hacia 1, así que el factor de crecimiento no es constante; iterar la cota de un paso con un cociente fijo es una extrapolación sin respaldo. La cota además ignora la construcción de instancias por el cruce, así que es holgada (Figura 12).</details>

<details><summary>8. Dos ejecuciones de un GA se detienen tras 100 generaciones con la misma mejor aptitud. La distancia de Hamming media de la ejecución A es de 0.3 bits; la de la ejecución B, de 6 bits. ¿Qué ejecución se beneficiaría más de 100 generaciones adicionales?</summary>La ejecución B. La ejecución A ha convergido prematuramente: la población es casi uniforme y solo la mutación puede producir material nuevo, lentamente; la ejecución B todavía conserva diversidad que el cruce puede explotar. A la ejecución A le beneficiaría más un reinicio con una tasa de mutación mayor o una población más grande que una continuación (Sección 6.3).</details>

<details><summary>9. Un GA para selección de características informa una exactitud en validación cruzada de 0.99 para el subconjunto seleccionado, frente a 0.96 para todas las características. Un revisor dice que el 0.99 no es una estimación del rendimiento de generalización. Explica la objeción y qué debería informarse en su lugar.</summary>El puntaje de validación cruzada era la función de aptitud: el GA examinó muchos subconjuntos y devolvió el de mayor puntaje, así que el 0.99 es un máximo sobre muchas estimaciones ruidosas y está sesgado hacia arriba (sesgo de selección, Ambroise y McLachlan). Debería informarse la exactitud del subconjunto seleccionado en un conjunto de prueba que nunca se usó durante la búsqueda, o una validación cruzada anidada en la que el GA se vuelve a ejecutar dentro de cada partición externa (Sección 8.1).</details>

<details><summary>10. El teorema de no hay almuerzo gratis establece que todos los optimizadores rinden igual cuando se promedian sobre todas las funciones objetivo posibles. ¿Significa esto que elegir entre un GA y el recocido simulado es arbitrario?</summary>No. El teorema se refiere al promedio sobre <em>todas</em> las funciones, la mayoría de las cuales no tienen estructura alguna. Los problemas reales tienen estructura (suavidad, separabilidad, bloques de construcción, cuencas regulares), y un método rinde bien en la clase cuya estructura explota. La elección debe justificarse por lo que se sabe del problema y verificarse empíricamente frente a la búsqueda aleatoria con igual presupuesto.</details>

<details><summary>11. Asocia cada método con su idea: CMA-ES, evolución diferencial, optimización por enjambre de partículas, recocido simulado, programación genética. Ideas: (a) diferencia escalada de dos miembros como paso de mutación; (b) aceptar una solución peor con probabilidad \(\exp(-\Delta/T)\); (c) árboles de expresiones como cromosomas; (d) adaptar la covarianza de la distribución de mutación; (e) velocidad atraída hacia el mejor personal y el mejor global.</summary>CMA-ES — (d); evolución diferencial — (a); optimización por enjambre de partículas — (e); recocido simulado — (b); programación genética — (c).</details>

<details><summary>12. En un GA de planificación de radioterapia, la aptitud es "dosis al tumor menos dosis a la médula espinal". Un plan obtiene una puntuación muy alta entregando una dosis enorme al tumor y una dosis a la médula justo por debajo de la tolerancia. ¿Qué ha ocurrido, y qué ilustra esto sobre el diseño de la aptitud?</summary>El GA ha optimizado exactamente la función que se le dio: sin un límite superior para la dosis al tumor ni una restricción sobre otros órganos, la aptitud lineal recompensa una dosis ilimitada. El algoritmo no hizo nada mal: la función de aptitud no expresaba lo que se quería. Las restricciones deben codificarse (penalizaciones, cotas o una formulación multiobjetivo), y el resultado debe ser inspeccionado por un ser humano antes de usarse (Secciones 3.2, 7 y 8.4).</details>

## Glosario

<dl class="gloss">
<dt>Alelo (allele)</dt><dd>El valor que contiene un gen en una posición dada (locus) de un cromosoma; 0 o 1 en un GA binario.</dd>
<dt>Bloque de construcción (building block)</dt><dd>Un esquema corto, de orden bajo y con aptitud por encima del promedio; la hipótesis de los bloques de construcción sostiene que un GA ensambla soluciones a partir de tales esquemas mediante el cruce.</dd>
<dt>Cromosoma, individuo (chromosome, individual)</dt><dd>La representación codificada completa de una solución candidata: una cadena de bits, un vector real o una permutación.</dd>
<dt>Cruce, recombinación (crossover, recombination)</dt><dd>Un operador que produce descendencia combinando genes de dos progenitores; de un punto, de dos puntos y uniforme para cadenas; aritmético, BLX-\(\alpha\) y SBX para vectores reales; cruce de orden para permutaciones.</dd>
<dt>Longitud definitoria δ(H) (defining length)</dt><dd>La distancia entre la primera y la última posición fija de un esquema; los esquemas largos tienen más probabilidad de ser cortados por el cruce de un punto.</dd>
<dt>Evolución diferencial (differential evolution)</dt><dd>Un algoritmo evolutivo de valores reales cuyo paso de mutación es la diferencia escalada de dos miembros de la población.</dd>
<dt>Diversidad (diversity)</dt><dd>Una medida de cuán distintos son los individuos de una población (distancia de Hamming media por pares, número de individuos distintos, dispersión alrededor del centroide); su colapso señala una convergencia prematura.</dd>
<dt>Elitismo (elitism)</dt><dd>Copiar sin cambios el mejor individuo o los pocos mejores a la generación siguiente, de modo que la mejor aptitud nunca disminuya.</dd>
<dt>Codificación, representación (encoding, representation)</dt><dd>La correspondencia entre soluciones candidatas y cromosomas; la función de decodificación lleva del genotipo al fenotipo.</dd>
<dt>Estrategia evolutiva (evolution strategy)</dt><dd>Un algoritmo evolutivo de valores reales que se apoya en la mutación gaussiana con tamaños de paso autoadaptados; CMA-ES adapta la matriz de covarianza completa.</dd>
<dt>Exploración / explotación (exploration / exploitation)</dt><dd>Muestrear regiones desconocidas / refinar regiones que ya se sabe que son buenas; la mutación y las poblaciones grandes exploran, la selección y el cruce explotan.</dd>
<dt>Función de aptitud (fitness function)</dt><dd>La función objetivo, que se desea maximizar, evaluada sobre el fenotipo; la única información sobre el problema que usa un GA.</dd>
<dt>Escalamiento de la aptitud (fitness scaling)</dt><dd>Una transformación de la aptitud bruta (escalamiento lineal, ranking) que controla la presión de selección con selección proporcional a la aptitud.</dd>
<dt>Generación (generation)</dt><dd>Una iteración del ciclo del GA: selección, cruce, mutación, reemplazo.</dd>
<dt>Programación genética (genetic programming)</dt><dd>Búsqueda evolutiva sobre programas o expresiones representados como árboles.</dd>
<dt>Genotipo / fenotipo (genotype / phenotype)</dt><dd>El cromosoma codificado sobre el que actúan los operadores / la solución decodificada que evalúa la función de aptitud.</dd>
<dt>Mutación (mutation)</dt><dd>Perturbación aleatoria de un cromosoma (inversión de bit, gaussiana, intercambio); la fuente de alelos no presentes en la población y la garantía de que todo punto es alcanzable.</dd>
<dt>Nichos, aptitud compartida (niching, fitness sharing)</dt><dd>Técnicas que permiten a una población ocupar varios óptimos a la vez penalizando las regiones abarrotadas.</dd>
<dt>Teorema de no hay almuerzo gratis (no-free-lunch theorem)</dt><dd>Promediados sobre todas las funciones objetivo posibles, todos los optimizadores rinden de forma idéntica; la superioridad en una clase de problemas proviene de explotar su estructura.</dd>
<dt>Orden de un esquema o(H) (order of a schema)</dt><dd>El número de posiciones fijas (distintas de *) en un esquema.</dd>
<dt>Optimización por enjambre de partículas (particle swarm optimisation)</dt><dd>Un método de enjambre en el que las partículas se mueven con velocidades atraídas hacia su mejor posición personal y hacia la mejor posición del enjambre.</dd>
<dt>Convergencia prematura (premature convergence)</dt><dd>Pérdida de diversidad de la población alrededor de una solución subóptima antes de que se haya encontrado el óptimo.</dd>
<dt>Selección por ruleta (roulette-wheel selection)</dt><dd>Selección proporcional a la aptitud: el individuo \(i\) se elige con probabilidad \(f_i / \sum_j f_j\).</dd>
<dt>Esquema (schema)</dt><dd>Una plantilla sobre {0, 1, *} que denota un conjunto de cromosomas; la unidad de análisis del teorema de los esquemas de Holland.</dd>
<dt>Presión de selección (selection pressure)</dt><dd>El grado en que la selección favorece a los individuos más aptos; se mide por el tiempo de toma de control o por las copias esperadas del mejor individuo.</dd>
<dt>Recocido simulado (simulated annealing)</dt><dd>Una búsqueda de un solo punto que acepta vecinos peores con probabilidad \(\exp(-\Delta/T)\) bajo una temperatura \(T\) decreciente.</dd>
<dt>Muestreo universal estocástico (stochastic universal sampling)</dt><dd>Selección proporcional a la aptitud con un solo giro y \(N\) punteros igualmente espaciados, que minimiza el error de muestreo.</dd>
<dt>Selección por torneo (tournament selection)</dt><dd>Elegir el mejor de \(k\) individuos extraídos al azar; \(k\) fija la presión de selección.</dd>
<dt>Envoltorio / filtro (wrapper / filter)</dt><dd>Selección de características que evalúa los subconjuntos ejecutando el clasificador / mediante un estadístico calculado sin el clasificador.</dd>
</dl>

## Referencias y lecturas recomendadas

<ol class="refs">
<li id="r1">Holland JH. <em>Adaptation in Natural and Artificial Systems</em>. Ann Arbor: University of Michigan Press; 1975. Reeditado en Cambridge, MA: MIT Press; 1992. <a href="https://doi.org/10.7551/mitpress/1090.001.0001">doi:10.7551/mitpress/1090.001.0001</a></li>
<li id="r2">Goldberg DE. <em>Genetic Algorithms in Search, Optimization and Machine Learning</em>. Reading, MA: Addison-Wesley; 1989. ISBN 978-0-201-15767-3. <a href="https://dl.acm.org/doi/book/10.5555/534133">dl.acm.org/doi/book/10.5555/534133</a> (sin DOI; registro de la ACM)</li>
<li id="r3">De Jong KA. <em>An Analysis of the Behavior of a Class of Genetic Adaptive Systems</em>. Tesis doctoral, University of Michigan; 1975. <a href="https://hdl.handle.net/2027.42/4507">hdl.handle.net/2027.42/4507</a> (sin DOI)</li>
<li id="r4">Mitchell M. <em>An Introduction to Genetic Algorithms</em>. Cambridge, MA: MIT Press; 1996. <a href="https://doi.org/10.7551/mitpress/3927.001.0001">doi:10.7551/mitpress/3927.001.0001</a></li>
<li id="r5">Eiben AE, Smith JE. <em>Introduction to Evolutionary Computing</em>. 2.ª ed. Berlin: Springer; 2015. <a href="https://doi.org/10.1007/978-3-662-44874-8">doi:10.1007/978-3-662-44874-8</a></li>
<li id="r6">Bäck T. <em>Evolutionary Algorithms in Theory and Practice</em>. New York: Oxford University Press; 1996. <a href="https://doi.org/10.1093/oso/9780195099713.001.0001">doi:10.1093/oso/9780195099713.001.0001</a></li>
<li id="r7">Whitley D. A genetic algorithm tutorial. <em>Stat Comput</em>. 1994;4(2):65–85. <a href="https://doi.org/10.1007/BF00175354">doi:10.1007/BF00175354</a></li>
<li id="r8">Blickle T, Thiele L. A comparison of selection schemes used in evolutionary algorithms. <em>Evol Comput</em>. 1996;4(4):361–394. <a href="https://doi.org/10.1162/evco.1996.4.4.361">doi:10.1162/evco.1996.4.4.361</a></li>
<li id="r9">Eshelman LJ, Schaffer JD. Real-coded genetic algorithms and interval-schemata. In: <em>Foundations of Genetic Algorithms 2</em>. Morgan Kaufmann; 1993:187–202. <a href="https://doi.org/10.1016/B978-0-08-094832-4.50018-0">doi:10.1016/B978-0-08-094832-4.50018-0</a></li>
<li id="r10">Larrañaga P, Kuijpers CMH, Murga RH, Inza I, Dizdarevic S. Genetic algorithms for the travelling salesman problem: a review of representations and operators. <em>Artif Intell Rev</em>. 1999;13(2):129–170. <a href="https://doi.org/10.1023/A:1006529012972">doi:10.1023/A:1006529012972</a></li>
<li id="r11">Srinivas M, Patnaik LM. Adaptive probabilities of crossover and mutation in genetic algorithms. <em>IEEE Trans Syst Man Cybern</em>. 1994;24(4):656–667. <a href="https://doi.org/10.1109/21.286385">doi:10.1109/21.286385</a></li>
<li id="r12">Rudolph G. Convergence analysis of canonical genetic algorithms. <em>IEEE Trans Neural Netw</em>. 1994;5(1):96–101. <a href="https://doi.org/10.1109/72.265964">doi:10.1109/72.265964</a></li>
<li id="r13">Sareni B, Krähenbühl L. Fitness sharing and niching methods revisited. <em>IEEE Trans Evol Comput</em>. 1998;2(3):97–106. <a href="https://doi.org/10.1109/4235.735432">doi:10.1109/4235.735432</a></li>
<li id="r14">Mühlenbein H, Schomisch M, Born J. The parallel genetic algorithm as function optimizer. <em>Parallel Comput</em>. 1991;17(6–7):619–632. <a href="https://doi.org/10.1016/S0167-8191(05)80052-3">doi:10.1016/S0167-8191(05)80052-3</a></li>
<li id="r15">Khuri S, Bäck T, Heitkötter J. The zero/one multiple knapsack problem and genetic algorithms. In: <em>Proc. 1994 ACM Symposium on Applied Computing</em>. 1994:188–193. <a href="https://doi.org/10.1145/326619.326694">doi:10.1145/326619.326694</a></li>
<li id="r16">Hansen N, Ostermeier A. Completely derandomized self-adaptation in evolution strategies. <em>Evol Comput</em>. 2001;9(2):159–195. <a href="https://doi.org/10.1162/106365601750190398">doi:10.1162/106365601750190398</a></li>
<li id="r17">Koza JR. Genetic programming as a means for programming computers by natural selection. <em>Stat Comput</em>. 1994;4(2):87–112. <a href="https://doi.org/10.1007/BF00175355">doi:10.1007/BF00175355</a></li>
<li id="r18">Storn R, Price K. Differential evolution — a simple and efficient heuristic for global optimization over continuous spaces. <em>J Glob Optim</em>. 1997;11(4):341–359. <a href="https://doi.org/10.1023/A:1008202821328">doi:10.1023/A:1008202821328</a></li>
<li id="r19">Kennedy J, Eberhart R. Particle swarm optimization. In: <em>Proc. ICNN'95</em>. 1995;4:1942–1948. <a href="https://doi.org/10.1109/ICNN.1995.488968">doi:10.1109/ICNN.1995.488968</a>. Aplicación a ECG: Melgani F, Bazi Y. Classification of electrocardiogram signals with support vector machines and particle swarm optimization. <em>IEEE Trans Inf Technol Biomed</em>. 2008;12(5):667–677. <a href="https://doi.org/10.1109/TITB.2008.923147">doi:10.1109/TITB.2008.923147</a></li>
<li id="r20">Kirkpatrick S, Gelatt CD, Vecchi MP. Optimization by simulated annealing. <em>Science</em>. 1983;220(4598):671–680. <a href="https://doi.org/10.1126/science.220.4598.671">doi:10.1126/science.220.4598.671</a></li>
<li id="r21">Wolpert DH, Macready WG. No free lunch theorems for optimization. <em>IEEE Trans Evol Comput</em>. 1997;1(1):67–82. <a href="https://doi.org/10.1109/4235.585893">doi:10.1109/4235.585893</a></li>
<li id="r22">Deb K, Pratap A, Agarwal S, Meyarivan T. A fast and elitist multiobjective genetic algorithm: NSGA-II. <em>IEEE Trans Evol Comput</em>. 2002;6(2):182–197. <a href="https://doi.org/10.1109/4235.996017">doi:10.1109/4235.996017</a></li>
<li id="r23">Siedlecki W, Sklansky J. A note on genetic algorithms for large-scale feature selection. <em>Pattern Recognit Lett</em>. 1989;10(5):335–347. <a href="https://doi.org/10.1016/0167-8655(89)90037-8">doi:10.1016/0167-8655(89)90037-8</a></li>
<li id="r24">Yang J, Honavar V. Feature subset selection using a genetic algorithm. <em>IEEE Intell Syst</em>. 1998;13(2):44–49. <a href="https://doi.org/10.1109/5254.671091">doi:10.1109/5254.671091</a></li>
<li id="r25">Kohavi R, John GH. Wrappers for feature subset selection. <em>Artif Intell</em>. 1997;97(1–2):273–324. <a href="https://doi.org/10.1016/S0004-3702(97)00043-X">doi:10.1016/S0004-3702(97)00043-X</a></li>
<li id="r26">Ambroise C, McLachlan GJ. Selection bias in gene extraction on the basis of microarray gene-expression data. <em>Proc Natl Acad Sci USA</em>. 2002;99(10):6562–6566. <a href="https://doi.org/10.1073/pnas.102102699">doi:10.1073/pnas.102102699</a></li>
<li id="r27">Friedrichs F, Igel C. Evolutionary tuning of multiple SVM parameters. <em>Neurocomputing</em>. 2005;64:107–117. <a href="https://doi.org/10.1016/j.neucom.2004.11.022">doi:10.1016/j.neucom.2004.11.022</a></li>
<li id="r28">Stanley KO, Miikkulainen R. Evolving neural networks through augmenting topologies. <em>Evol Comput</em>. 2002;10(2):99–127. <a href="https://doi.org/10.1162/106365602320169811">doi:10.1162/106365602320169811</a></li>
<li id="r29">Ezzell GA. Genetic and geometric optimization of three-dimensional radiation therapy treatment planning. <em>Med Phys</em>. 1996;23(3):293–305. <a href="https://doi.org/10.1118/1.597660">doi:10.1118/1.597660</a></li>
<li id="r30">Jensen JH. A graph-based genetic algorithm and generative model/Monte Carlo tree search for the exploration of chemical space. <em>Chem Sci</em>. 2019;10(12):3567–3572. <a href="https://doi.org/10.1039/C8SC05372C">doi:10.1039/C8SC05372C</a></li>
<li id="r31">Brown N, Fiscato M, Segler MHS, Vaucher AC. GuacaMol: benchmarking models for de novo molecular design. <em>J Chem Inf Model</em>. 2019;59(3):1096–1108. <a href="https://doi.org/10.1021/acs.jcim.8b00839">doi:10.1021/acs.jcim.8b00839</a></li>
<li id="r32">Jones DT. De novo protein design using pairwise potentials and a genetic algorithm. <em>Protein Sci</em>. 1994;3(4):567–574. <a href="https://doi.org/10.1002/pro.5560030405">doi:10.1002/pro.5560030405</a></li>
<li id="r33">Nasiri JA, Naghibzadeh M, Yazdi HS, Naghibzadeh B. ECG arrhythmia classification with support vector machines and genetic algorithm. In: <em>2009 Third UKSim European Symposium on Computer Modeling and Simulation</em>. IEEE; 2009:187–192. <a href="https://doi.org/10.1109/EMS.2009.39">doi:10.1109/EMS.2009.39</a></li>
<li id="r34">Li L, Weinberg CR, Darden TA, Pedersen LG. Gene selection for sample classification based on gene expression data: study of sensitivity to choice of parameters of the GA/KNN method. <em>Bioinformatics</em>. 2001;17(12):1131–1142. <a href="https://doi.org/10.1093/bioinformatics/17.12.1131">doi:10.1093/bioinformatics/17.12.1131</a></li>
<li id="r35">Street WN, Wolberg WH, Mangasarian OL. Nuclear feature extraction for breast tumor diagnosis. <em>Proc SPIE</em>. 1993;1905:861–870. <a href="https://doi.org/10.1117/12.148698">doi:10.1117/12.148698</a></li>
<li id="r36">R Core Team. <em>R: A Language and Environment for Statistical Computing</em>. R Foundation for Statistical Computing, Vienna. <a href="https://www.R-project.org/">https://www.R-project.org/</a> (software; sin DOI)</li>
<li id="r37">Leisch F, Dimitriadou E. mlbench: Machine Learning Benchmark Problems. Paquete de R. <a href="https://doi.org/10.32614/CRAN.package.mlbench">doi:10.32614/CRAN.package.mlbench</a></li>
<li id="r38">Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em>. 4.ª ed. New York: Springer; 2002. <a href="https://doi.org/10.1007/978-0-387-21706-2">doi:10.1007/978-0-387-21706-2</a> (la referencia para las funciones de modelado de R <code>knn</code> y <code>svm</code> usadas aquí)</li>
</ol>

<h4>Libros de texto</h4>
<ul>
<li>Russell S, Norvig P. <em>Artificial Intelligence: A Modern Approach</em>. 4.ª ed. Pearson; 2021. — El Capítulo 4 cubre el escalamiento de colinas, el recocido simulado, los algoritmos genéticos y la búsqueda local en espacios continuos.</li>
<li>Eiben AE, Smith JE. <em>Introduction to Evolutionary Computing</em>. 2.ª ed. Springer; 2015. — El libro de texto estándar; los ocho componentes de la Sección 3 siguen su organización.</li>
<li>Goldberg DE. <em>Genetic Algorithms in Search, Optimization and Machine Learning</em>. Addison-Wesley; 1989. — El Capítulo 1 contiene el ejemplo resuelto a mano de la Sección 4; el Capítulo 2, el teorema de los esquemas.</li>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2.ª ed. Springer; 2021. — El complemento en R de todo el curso; PDF gratuito en statlearning.com.</li>
<li>Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2.ª ed. Springer; 2009. — La Sección 7.10.2 ("The wrong and right way to do cross-validation") formaliza el sesgo de selección de la Sección 8.1.</li>
</ul>

<h4>Recursos en línea</h4>
<ul>
<li>CRAN Task View: Optimization and Mathematical Programming — <a href="https://cran.r-project.org/view=Optimization">cran.r-project.org/view=Optimization</a>; paquetes de R para optimización global, incluidos los paquetes <code>GA</code> y <code>DEoptim</code>, útiles una vez que se comprende la mecánica de este notebook (no se usan aquí, ya que el notebook implementa el GA desde cero).</li>
<li>Página de CMA-ES de Nikolaus Hansen — <a href="http://cma-es.github.io">cma-es.github.io</a>; tutorial e implementaciones de referencia.</li>
<li>Benchmark GuacaMol — <a href="https://github.com/BenevolentAI/guacamol">github.com/BenevolentAI/guacamol</a>; el conjunto de pruebas de diseño molecular de novo de la Sección 8.4, con el GA basado en grafos entre sus líneas base.</li>
<li>UCI Machine Learning Repository — <a href="https://archive.ics.uci.edu">archive.ics.uci.edu</a>; fuente original de los datos de cáncer de mama.</li>
</ul>
